# 📱 Customer Churn Prediction — End-to-End Machine Learning Project

**Dataset:** Cell2Cell Telecom Churn (51,047 labelled customers + 20,000 unlabelled holdout customers)
**Goal:** Estimate each customer's probability of cancelling, so a retention team can prioritise its budget
**Author:** Ali Elshafey — Projektarbeit, Fachhochschule Dortmund (supervisor: Prof. Dr. Nadia Bauer)

Every section explains *what* is done, *why*, and *how to read the output*.

---

## 🗺️ Project Roadmap

| # | Section | What happens |
|---|---------|-------------|
| 1 | Setup & Imports | Load libraries and both datasets |
| 2 | Data Inspection | Understand shape, types, missing values, target |
| 3 | Data Cleaning | Fix every data quality issue, documented step-by-step |
| 4 | Exploratory Data Analysis | Visualise churn patterns and drivers |
| 5 | Features, Feature Engineering & Selection | Prepare X and y, add ratio features, drop redundant columns |
| 6 | Preprocessing Pipeline | Impute, encode, scale — with zero data leakage |
| 7 | Train / Validation Split | Stratified 80/20 split of training data only |
| 8 | Model Comparison | Logistic Regression, Random Forest, XGBoost, LightGBM |
| 9 | Class Imbalance Handling | Class weights + threshold tuning + SMOTE |
| 10 | Hyperparameter Tuning & Model Selection | RandomizedSearchCV, GridSearchCV, soft-voting ensemble |
| 11 | Model Interpretation | Feature importance + permutation importance + SHAP |
| 12 | Holdout Predictions | One-time scoring of completely unseen customers |
| 13 | Summary & Recommendations | Business conclusions and lessons |

> ⚠️ **Ground rule:** `cell2cellholdout.csv` is **never touched** until Section 12.
> All development, fitting, tuning, and threshold selection uses only `cell2celltrain.csv`.

---
## Section 1 — Setup, Imports & Data Loading

### 📖 What and Why

Before any analysis we need to:
1. **Install extra libraries** (only needed on Google Colab — locally use `pip install -r requirements.txt`)
2. **Import everything at the top** — so all dependencies are visible upfront
3. **Load both CSV files** from the `data/` folder (see the README for the download link)

In [ ]:
# Only needed on Google Colab — locally, run:  pip install -r requirements.txt
import sys
if 'google.colab' in sys.modules:
    %pip install -q xgboost lightgbm shap imbalanced-learn

In [ ]:
# ── Standard data libraries ───────────────────────────────────
import warnings
warnings.filterwarnings('ignore')
import numpy  as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')

# ── Scikit-learn: model selection ─────────────────────────────
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                      RandomizedSearchCV, cross_val_score)
# ── Scikit-learn: preprocessing ───────────────────────────────
from sklearn.pipeline    import Pipeline
from sklearn.compose     import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute      import SimpleImputer

# ── Scikit-learn: classifiers ─────────────────────────────────
from sklearn.dummy             import DummyClassifier
from sklearn.linear_model      import LogisticRegression
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import (RandomForestClassifier,
                                        GradientBoostingClassifier,
                                        ExtraTreesClassifier)
from sklearn.neural_network    import MLPClassifier

# ── Advanced gradient boosting ─────────────────────────────────
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier

# ── Evaluation ─────────────────────────────────────────────────
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, roc_curve,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import permutation_importance

# ── Class imbalance ────────────────────────────────────────────
from imblearn.over_sampling import SMOTE

# ── SHAP explainability ────────────────────────────────────────
import shap

# ── Hyperparameter distributions ──────────────────────────────
from scipy.stats import randint, uniform

print(f"✅ All libraries imported  |  pandas {pd.__version__}  |  numpy {np.__version__}")

In [ ]:
# Locate the project folders (works when run from the repo root or from notebooks/)
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR   = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

TRAIN_PATH   = DATA_DIR / 'cell2celltrain.csv'
HOLDOUT_PATH = DATA_DIR / 'cell2cellholdout.csv'

if not TRAIN_PATH.exists():
    try:
        # On Google Colab: upload cell2celltrain.csv and cell2cellholdout.csv
        from google.colab import files
        DATA_DIR.mkdir(exist_ok=True)
        for name, content in files.upload().items():
            (DATA_DIR / name).write_bytes(content)
    except ImportError:
        raise FileNotFoundError(
            "Download cell2celltrain.csv and cell2cellholdout.csv from Kaggle "
            "and place them in the data/ folder (see README).")

print(f"✅ Data folder: {DATA_DIR}")

In [ ]:
# Load into DataFrames
# df_train   → our working dataset for ALL development
# df_holdout → locked away until the very last section

df_train   = pd.read_csv(TRAIN_PATH)
df_holdout = pd.read_csv(HOLDOUT_PATH)

print(f"Training dataset : {df_train.shape[0]:>6,} rows × {df_train.shape[1]} columns")
print(f"Holdout  dataset : {df_holdout.shape[0]:>6,} rows × {df_holdout.shape[1]} columns")
print()
print("NOTE: df_holdout has no Churn labels — it is a true prediction holdout.")
print("      We will use it exactly ONCE at the very end of this notebook.")

In [ ]:
df_train.describe()

### 📊 What the output tells you

- **Training set ~51,047 rows** — your entire working dataset for model development
- **Holdout set ~20,000 rows** — completely unseen, no labels, used only for the final test

Having a genuinely untouched holdout is what makes your final evaluation trustworthy.
If you accidentally trained on holdout data, your reported performance would be **optimistically biased**.

---
## Section 2 — Data Inspection

### 📖 What and Why

Before touching any data you must **understand it**. Check:
- **Shape** — rows and columns
- **Column names & dtypes** — what kind of information is here?
- **Missing values** — which features have gaps and how many?
- **Duplicates** — are any customers listed twice?
- **Target column** — what exactly are we predicting?
- **Categorical cardinality** — how many unique values per column?

> 💡 **Learning tip:** Skipping inspection is the #1 beginner mistake.
> Never clean or model data you haven't looked at first.

In [ ]:
# 2.1 — First rows and basic shape
print("=" * 60)
print("TRAINING DATASET — first 5 rows")
print("=" * 60)
display(df_train.head())
print(f"\nShape: {df_train.shape[0]:,} rows  ×  {df_train.shape[1]} columns")

**What to notice in the first rows:**
- `CustomerID` — a unique row identifier, NOT a feature (must be removed)
- `Churn` — our target: `'Yes'` or `'No'` (will be encoded to 1/0)
- Mix of numeric columns (call counts, charges) and text/categorical columns

In [ ]:
# 2.2 — Data types for every column
print("Column data types:\n")
dtype_df = pd.DataFrame({
    'Column' : df_train.dtypes.index,
    'Dtype'  : df_train.dtypes.values.astype(str),
    'Sample' : [str(df_train[c].dropna().iloc[0])
                if df_train[c].notna().any() else 'NaN'
                for c in df_train.columns]
})
display(dtype_df)

In [ ]:
df_train.isnull().sum()

**What to notice:**
- `HandsetPrice` is `object` dtype because it contains `'Unknown'` strings — needs fixing to numeric
- `CreditRating` is `object` with values like `'1-Highest'`…`'7-Lowest'` — needs ordinal extraction
- Several columns are binary Yes/No strings — will be handled by `OrdinalEncoder` in the pipeline

In [ ]:
# 2.3 — Missing values report
def missing_report(df, name="Dataset"):
    total  = df.isnull().sum()
    pct    = (total / len(df) * 100).round(2)
    report = pd.DataFrame({'Missing Count': total, 'Missing %': pct})
    report = report[report['Missing Count'] > 0].sort_values('Missing Count', ascending=False)
    print(f"\n{name}  —  {len(report)} columns with missing values  ({len(df):,} rows total)\n")
    display(report)
    return report

miss_train   = missing_report(df_train,   "TRAINING SET")
miss_holdout = missing_report(df_holdout, "HOLDOUT SET  (for reference only)")

**How to read the missing value report:**
- All missing percentages are below 5% — acceptable to impute, not drop
- `AgeHH1/AgeHH2` (~1.8% missing) and `PercChangeMinutes/Revenues` (~0.7%) are the worst
- The same columns are missing in holdout — confirming our imputation strategy must generalise
- `Churn` is missing in holdout — **expected**, that's the column we need to predict

**Decision:** Use **median imputation** for numeric and **constant `'missing'` fill** for categorical.
Imputation will happen **inside the pipeline** to prevent data leakage.

In [ ]:
# 2.4 — Duplicate rows check
print(f"Duplicate rows in training set : {df_train.duplicated().sum()}")
print(f"Duplicate rows in holdout set  : {df_holdout.duplicated().sum()}")

# 2.5 — Target column distribution
print("\n" + "=" * 50)
print("TARGET COLUMN: 'Churn'")
print("=" * 50)
churn_counts = df_train['Churn'].value_counts()
churn_pct    = df_train['Churn'].value_counts(normalize=True) * 100

summary = pd.DataFrame({'Count': churn_counts, 'Percentage (%)': churn_pct.round(2)})
display(summary)

print(f"\n⚠️  Class imbalance detected!")
print(f"   Churners     (Yes): {churn_counts['Yes']:>6,}  ({churn_pct['Yes']:.1f}%)")
print(f"   Non-churners  (No): {churn_counts['No']:>6,}  ({churn_pct['No']:.1f}%)")
print(f"   Imbalance ratio   :  1 : {churn_counts['No']/churn_counts['Yes']:.1f}")
print()
print("A naive model predicting 'No Churn' always would get")
print(f"{churn_pct['No']:.1f}% accuracy — but catch ZERO real churners!")

In [ ]:
# 2.6 — Inspect all categorical columns
cat_cols = df_train.select_dtypes(include=['object','string']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'Churn']

print(f"Found {len(cat_cols)} categorical columns (excluding target):\n")
for col in cat_cols:
    uniques = sorted(df_train[col].dropna().unique().tolist())
    print(f"  {col:<30}  {len(uniques):>3} unique values  →  {uniques[:8]}")

**Key observations:**
- Most binary columns have only `['No', 'Yes']` — will be ordinal-encoded as 0/1
- `CreditRating`: 7 ordered levels `'1-Highest'`…`'7-Lowest'` — extract numeric rank
- `HandsetPrice`: numeric values plus `'Unknown'` — convert to float after replacing Unknown with NaN
- `ServiceArea`: 500+ unique US area codes — too high cardinality, will be dropped
- `PrizmCode`, `Occupation`, `MaritalStatus`: manageable categories, keep them

---
## Section 3 — Data Cleaning

### 📖 What and Why

Every cleaning step needs a reason. We fix:
1. **Remove `CustomerID`** — identifier, no predictive value
2. **Fix `HandsetPrice`** — numeric disguised as string because of `'Unknown'`
3. **Extract ordinal rank from `CreditRating`** — `'1-Highest'` → `1`
4. **Drop `NotNewCellphoneUser`** — perfectly redundant with `NewCellphoneUser`
5. **Drop `ServiceArea`** — 500+ unique codes, high-cardinality noise
6. **Remove duplicates**

> ⚠️ **We write a `clean()` function** so we can apply identical steps to holdout later.
> This ensures consistency and prevents copy-paste errors.

In [ ]:
import pandas as pd
import numpy as np

# Data dictionary: type, explanation, value range and missing values per column
df = df_train.copy()

# Short explanation of every column
explanations = {
    "CustomerID": "Unique customer identifier; not used as predictive feature",
    "Churn": "Target variable indicating whether the customer cancelled the contract",
    "MonthlyRevenue": "Average monthly revenue generated by the customer",
    "MonthlyMinutes": "Average number of monthly call minutes",
    "TotalRecurringCharge": "Monthly recurring charge paid by the customer",
    "DirectorAssistedCalls": "Number of calls assisted by an operator",
    "OverageMinutes": "Minutes used beyond the included plan allowance",
    "RoamingCalls": "Number of roaming calls",
    "PercChangeMinutes": "Percentage change in monthly minutes",
    "PercChangeRevenues": "Percentage change in monthly revenue",
    "DroppedCalls": "Number of dropped calls",
    "BlockedCalls": "Number of blocked calls",
    "UnansweredCalls": "Number of unanswered calls",
    "CustomerCareCalls": "Number of calls to customer care",
    "ThreewayCalls": "Number of three-way calls",
    "ReceivedCalls": "Number of received calls",
    "OutboundCalls": "Number of outgoing calls",
    "InboundCalls": "Number of incoming calls",
    "PeakCallsInOut": "Number of calls during peak hours",
    "OffPeakCallsInOut": "Number of calls during off-peak hours",
    "DroppedBlockedCalls": "Combined number of dropped and blocked calls",
    "CallForwardingCalls": "Number of call forwarding calls",
    "CallWaitingCalls": "Number of call waiting calls",
    "MonthsInService": "Number of months the customer has been with the provider",
    "UniqueSubs": "Number of unique subscribers linked to the account",
    "ActiveSubs": "Number of active subscribers linked to the account",
    "ServiceArea": "Customer service area or region",
    "Handsets": "Number of handsets used by the customer",
    "HandsetModels": "Number of different handset models used",
    "CurrentEquipmentDays": "Number of days the current equipment has been in use",
    "AgeHH1": "Age of the first household member",
    "AgeHH2": "Age of the second household member",
    "ChildrenInHH": "Indicates whether children live in the household",
    "HandsetRefurbished": "Indicates whether the handset is refurbished",
    "HandsetWebCapable": "Indicates whether the handset supports web access",
    "TruckOwner": "Indicates whether the customer owns a truck",
    "RVOwner": "Indicates whether the customer owns a recreational vehicle",
    "Homeownership": "Customer home ownership status",
    "BuysViaMailOrder": "Indicates whether the customer buys via mail order",
    "RespondsToMailOffers": "Indicates whether the customer responds to mail offers",
    "OptOutMailings": "Indicates whether the customer opted out of mailings",
    "NonUSTravel": "Indicates whether the customer travels outside the US",
    "OwnsComputer": "Indicates whether the customer owns a computer",
    "HasCreditCard": "Indicates whether the customer has a credit card",
    "RetentionCalls": "Number of retention-related calls",
    "RetentionOffersAccepted": "Number of accepted retention offers",
    "NewCellphoneUser": "Indicates whether the customer is a new cellphone user",
    "NotNewCellphoneUser": "Indicates whether the customer is not a new cellphone user",
    "ReferralsMadeBySubscriber": "Number of referrals made by the customer",
    "IncomeGroup": "Income group of the customer",
    "OwnsMotorcycle": "Indicates whether the customer owns a motorcycle",
    "AdjustmentsToCreditRating": "Number of adjustments made to credit rating",
    "HandsetPrice": "Price of the customer handset",
    "MadeCallToRetentionTeam": "Indicates whether the customer contacted the retention team",
    "CreditRating": "Customer credit rating category",
    "PrizmCode": "Customer lifestyle or market segmentation group",
    "Occupation": "Customer occupation category",
    "MaritalStatus": "Customer marital status"
}

def get_type(series):
    if pd.api.types.is_numeric_dtype(series):
        return "metric"
    else:
        return "nominal"

def get_values_or_range(series):
    s = series.dropna()

    if len(s) == 0:
        return "Only missing values"

    if pd.api.types.is_numeric_dtype(s):
        return f"[{s.min()} - {s.max()}]"

    unique_vals = s.astype(str).unique()
    unique_vals = sorted(unique_vals)

    if len(unique_vals) <= 6:
        return ", ".join(unique_vals)
    else:
        return ", ".join(unique_vals[:6]) + ", ..."

description_table = pd.DataFrame({
    "Name": df.columns,
    "Type": [get_type(df[col]) for col in df.columns],
    "Explanation": [explanations.get(col, "Customer-related feature used for churn prediction") for col in df.columns],
    "Values / Range": [get_values_or_range(df[col]) for col in df.columns],
    "Missing Values": [df[col].isna().sum() for col in df.columns],
    "Missing %": [(df[col].isna().mean() * 100).round(2) for col in df.columns]
})

description_table

In [ ]:
def clean(df, is_train=True):
    """
    Clean a Cell2Cell dataset identically for both train and holdout.

    Parameters
    ----------
    df       : raw DataFrame
    is_train : if True, encode the Churn target column

    Returns
    -------
    df_clean : cleaned DataFrame
    log      : list of actions taken
    """
    df  = df.copy()   # never modify the original DataFrame
    log = []

    # Step 1: Remove unique identifier — no predictive value
    # If left in, a model might memorise IDs and fail on new data
    if 'CustomerID' in df.columns:
        df.drop(columns=['CustomerID'], inplace=True)
        log.append("✅ Dropped CustomerID — unique row identifier")

    # Step 2: Encode target (training set only)
    # Scikit-learn classifiers expect numeric targets (0/1), not strings
    if is_train and 'Churn' in df.columns:
        df['Churn'] = (df['Churn'] == 'Yes').astype(int)
        log.append("✅ Encoded Churn: Yes→1, No→0")

    # Step 3: Fix HandsetPrice
    # The string 'Unknown' prevented pandas from reading this as float.
    # Replace with NaN first, then convert. NaN is handled by imputer later.
    if 'HandsetPrice' in df.columns:
        df['HandsetPrice'] = pd.to_numeric(
            df['HandsetPrice'].replace('Unknown', float('nan')),
            errors='coerce'
        )
        log.append("✅ HandsetPrice: 'Unknown' → NaN, cast to float")

    # Step 4: Extract ordinal rank from CreditRating
    # '1-Highest' → 1, '2-High' → 2, …, '7-Lowest' → 7
    # This preserves the ORDER (ordinal information) as a numeric feature
    if 'CreditRating' in df.columns:
        df['CreditRating'] = (
            df['CreditRating'].astype(str)
            .str.extract(r'^(\d)')[0]
            .astype(float)
        )
        log.append("✅ CreditRating: extracted rank 1–7 (1=best, 7=worst)")

    # Step 5: Drop redundant column
    # NotNewCellphoneUser = 1 − NewCellphoneUser by definition.
    # Including both creates perfect negative correlation → misleads linear models
    if 'NotNewCellphoneUser' in df.columns:
        df.drop(columns=['NotNewCellphoneUser'], inplace=True)
        log.append("✅ Dropped NotNewCellphoneUser — redundant with NewCellphoneUser")

    # Step 6: Drop high-cardinality ServiceArea
    # 500+ unique US area codes. Any encoder would create 500 columns (OHE)
    # or meaningless ordinals. Neither adds signal for this dataset size.
    if 'ServiceArea' in df.columns:
        df.drop(columns=['ServiceArea'], inplace=True)
        log.append("✅ Dropped ServiceArea — 500+ unique codes, high-cardinality noise")

    # Step 7: Remove duplicates
    n_before = len(df)
    df.drop_duplicates(inplace=True)
    log.append(f"✅ Duplicates removed: {n_before - len(df)}")

    return df, log


# Apply cleaning to training data
df_clean, cleaning_log = clean(df_train, is_train=True)

print("CLEANING LOG")
print("-" * 55)
for entry in cleaning_log:
    print(f"  {entry}")
print(f"\nShape before cleaning : {df_train.shape}")
print(f"Shape after  cleaning : {df_clean.shape}")

**How to read the output:**
- Each ✅ confirms one cleaning step completed
- Shape goes from 58 → **55 columns** (removed 3: CustomerID, NotNewCellphoneUser, ServiceArea)
- All rows are preserved — we never drop rows for missing values
- Missing values still exist in the cleaned data — they are handled in the pipeline (Section 6)

> **Why not impute now?** If we imputed here (before the train/val split), we'd use information from
> the validation rows to fill training gaps. That's **data leakage** — it makes your model appear
> better than it really is. Imputing inside the pipeline solves this perfectly.

In [ ]:
# Verify the cleaned dataset
print("Sample of cleaned data:")
display(df_clean.head(3))

print("\nRemaining missing values (will be handled inside pipeline):")
remaining = df_clean.isnull().sum()
display(remaining[remaining > 0].sort_values(ascending=False))

print(f"\nTarget distribution:")
print(df_clean['Churn'].value_counts())

---
## Section 4 — Exploratory Data Analysis (EDA)

### 📖 What and Why

EDA means **understanding the story inside the data** before modelling.
Good EDA answers:
- Is there class imbalance?
- Which features separate churners from non-churners?
- Are any features correlated with each other?
- Are there patterns in how churners behave differently?

> 💡 **Learning tip:** EDA directly informs your modelling decisions.
> Insights here tell you which features matter, how to handle outliers,
> and how serious the imbalance problem is.

In [ ]:
# 4.1 — Class distribution visualisation
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

churn_counts = df_clean['Churn'].value_counts()
churn_pct    = churn_counts / len(df_clean) * 100

bars = axes[0].bar(['No Churn (0)', 'Churn (1)'],
                   churn_counts.values,
                   color=['#3B82F6', '#EF4444'], edgecolor='white', linewidth=1.5)
for bar, val, pct in zip(bars, churn_counts.values, churn_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 300,
                 f'{val:,}\n({pct:.1f}%)', ha='center', fontsize=11, fontweight='bold')
axes[0].set_title('Churn Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of customers')
axes[0].set_ylim(0, churn_counts.max() * 1.25)

axes[1].pie(churn_counts.values,
            labels=['No Churn', 'Churn'],
            autopct='%1.1f%%',
            colors=['#3B82F6', '#EF4444'],
            startangle=90,
            wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Churn Proportion', fontsize=13, fontweight='bold')

plt.suptitle('Target Variable: Class Imbalance Check', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"Churn rate: {churn_pct[1]:.1f}%  →  moderate class imbalance confirmed")

**What this tells you:**
- **28.8% churn rate** — moderate imbalance
- A model always predicting "No Churn" would score 71.2% accuracy but find **zero churners**
- We must use: (1) class-weighted models, (2) SMOTE, (3) threshold optimisation, (4) Recall/AUC metrics

In [ ]:
# Get all numeric columns from df_clean

numeric_columns = df_clean.select_dtypes(include=['number']).columns.tolist()

print(numeric_columns)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style='white')
plt.rcParams['figure.dpi'] = 140

# Colors
C_CHURN = '#FF6B6B'
C_STAY  = '#4ECDC4'
C_BG    = '#FAFAFA'

# Prepare data
churn_rate = df_clean['Churn'].mean() * 100

# ══════════════════════════════════════════════════════════════════
# DIAGRAM 2 — Usage & Service Quality (KDE + Bubble + Stacked)
# ══════════════════════════════════════════════════════════════════

fig = plt.figure(figsize=(18, 6), facecolor=C_BG)
gs = gridspec.GridSpec(1, 3, wspace=0.3)

# ── 2A — KDE: PercChangeMinutes ──────────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1.set_facecolor(C_BG)
for churn_val, color, label in [(0, C_STAY, 'Stayed'), (1, C_CHURN, 'Churned')]:
    data = df_clean.loc[df_clean['Churn']==churn_val, 'PercChangeMinutes'].dropna()
    data = data.clip(lower=data.quantile(0.02), upper=data.quantile(0.98))
    data.plot.kde(ax=ax1, color=color, linewidth=2.5, label=label, alpha=0.8)

ax1.axvline(0, color='black', linewidth=0.8, alpha=0.4, linestyle=':')
ax1.set_xlabel('Minutes Change (%)', fontsize=11)
ax1.set_ylabel('Density', fontsize=11)
ax1.set_title('Usage Change Distribution', fontsize=12, fontweight='bold', pad=10)
ax1.legend(fontsize=10, framealpha=0.9)
ax1.grid(alpha=0.3)
ax1.set_xlim(-300, 600)

# ── 2B — Bubble Dot Plot: CustomerCareCalls ──────────────────────
ax2 = fig.add_subplot(gs[1])
ax2.set_facecolor(C_BG)
df_clean['_care'] = pd.cut(df_clean['CustomerCareCalls'], bins=[-0.1,0,1,3,5,10,999],
                     labels=['0','1','2-3','4-5','6-10','10+'])
care_c = df_clean.groupby('_care')['Churn'].mean() * 100
care_n = df_clean.groupby('_care').size()

colors = [C_CHURN if v > churn_rate else C_STAY for v in care_c.values]
sizes = (care_n / care_n.max() * 400 + 80).values

for i, (val, sz, col) in enumerate(zip(care_c.values, sizes, colors)):
    ax2.scatter(val, i, s=sz, color=col, alpha=0.8, edgecolor='white', linewidth=2, zorder=3)
    ax2.text(val+0.8, i, f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
    ax2.plot([0, val], [i, i], color=col, linewidth=1, alpha=0.3)

ax2.axvline(churn_rate, color='#888', linestyle='--', linewidth=1.5, alpha=0.6,
            label=f'Avg {churn_rate:.1f}%')
ax2.set_yticks(range(len(care_c)))
ax2.set_yticklabels(care_c.index, fontsize=10)
ax2.set_xlabel('Churn Rate (%)', fontsize=11)
ax2.set_title('Care Calls → Churn',
              fontsize=12, fontweight='bold', pad=10)
ax2.legend(fontsize=9)
ax2.grid(axis='x', alpha=0.3)

# ── 2C — Stacked Horizontal Bars: DroppedBlockedCalls ────────────
ax3 = fig.add_subplot(gs[2])
ax3.set_facecolor(C_BG)
df_clean['_drop'] = pd.cut(df_clean['DroppedBlockedCalls'].fillna(0),
    bins=[-0.1,0,5,10,20,999], labels=['0','1-5','6-10','11-20','20+'])
cross = pd.crosstab(df_clean['_drop'], df_clean['Churn'], normalize='index') * 100

cross.plot.barh(stacked=True, ax=ax3, color=[C_STAY, C_CHURN],
                edgecolor='white', linewidth=0.5, width=0.7)

for i, (idx, row) in enumerate(cross.iterrows()):
    churn_pct = row[1]
    ax3.text(100+1, i, f'{churn_pct:.1f}%', va='center', fontsize=10,
             fontweight='bold', color=C_CHURN)

ax3.set_xlabel('Proportion (%)', fontsize=11)
ax3.set_title('Dropped+Blocked Calls\n(churn % shown on right)',
              fontsize=12, fontweight='bold', pad=10)
ax3.legend(['Stayed','Churned'], fontsize=9, loc='upper right')
ax3.grid(axis='x', alpha=0.3)
ax3.set_xlim(0, 115)

fig.suptitle('Diagram 2 — Usage Behavior & Service Quality',
             fontsize=15, fontweight='bold', y=1.04, color='#2D3748')
plt.tight_layout()
plt.show()

# Cleanup temp columns
df_clean.drop(columns=['_care','_drop'], inplace=True, errors='ignore')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style='white')
plt.rcParams['figure.dpi'] = 140

C_CHURN = '#FF6B6B'
C_STAY  = '#4ECDC4'
C_BG    = '#FAFAFA'
churn_rate = df_clean['Churn'].mean() * 100


# ══════════════════════════════════════════════════════════════
# DIAGRAM — Churn by Device Age (Lollipop)
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5), facecolor=C_BG)
ax.set_facecolor(C_BG)

df_clean['_dev'] = pd.cut(df_clean['CurrentEquipmentDays'],
    bins=[-1, 90, 180, 365, 730, 9999],
    labels=['< 3 months', '3-6 months', '6-12 months', '1-2 years', '2+ years'])
dev_c = df_clean.groupby('_dev')['Churn'].mean() * 100
colors = [C_CHURN if v > churn_rate else C_STAY for v in dev_c.values]

for i, (name, val) in enumerate(dev_c.items()):
    ax.plot(val, i, 'o', markersize=13, color=colors[i], zorder=3)
    ax.hlines(y=i, xmin=0, xmax=val, color=colors[i], linewidth=2.5, alpha=0.6)
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

ax.axvline(churn_rate, color='#888', linestyle='--', linewidth=1.5, alpha=0.6)
ax.set_yticks(range(len(dev_c)))
ax.set_yticklabels(dev_c.index, fontsize=10)
ax.set_xlabel('Churn Rate (%)', fontsize=11)
ax.set_title('Churn by Device Age', fontsize=13, fontweight='bold', pad=10)
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
df_clean.drop(columns=['_dev'], inplace=True, errors='ignore')


# ══════════════════════════════════════════════════════════════
# DIAGRAM — Risk Matrix: Device Age × Tenure (Heatmap)
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 6), facecolor=C_BG)
ax.set_facecolor(C_BG)

df_clean['_dg'] = pd.cut(df_clean['CurrentEquipmentDays'],
    bins=[-1, 180, 365, 730, 9999], labels=['<6m', '6-12m', '1-2yr', '2yr+'])
df_clean['_tg'] = pd.cut(df_clean['MonthsInService'],
    bins=[-1, 12, 24, 48, 999], labels=['<1yr', '1-2yr', '2-4yr', '4yr+'])
heat = df_clean.groupby(['_dg', '_tg'])['Churn'].mean().unstack() * 100

sns.heatmap(heat, annot=True, fmt='.1f', cmap='RdYlGn_r', center=churn_rate,
            linewidths=2, linecolor='white', ax=ax,
            cbar_kws={'label': 'Churn %', 'shrink': 0.8},
            annot_kws={'fontsize': 12, 'fontweight': 'bold'})
ax.set_xlabel('Customer Tenure', fontsize=11)
ax.set_ylabel('Device Age', fontsize=11)
ax.set_title('Risk Matrix\nDevice Age × Tenure', fontsize=13, fontweight='bold', pad=10)
plt.tight_layout()
plt.show()
df_clean.drop(columns=['_dg', '_tg'], inplace=True, errors='ignore')


# ══════════════════════════════════════════════════════════════
# DIAGRAM — Churn by Tenure (Lollipop)
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5), facecolor=C_BG)
ax.set_facecolor(C_BG)

df_clean['_ten'] = pd.cut(df_clean['MonthsInService'],
    bins=[-1, 6, 12, 24, 48, 999],
    labels=['0-6m (New)', '6-12m (Early)', '12-24m (Mid)',
            '24-48m (Estab.)', '48m+ (Loyal)'])
ten_c = df_clean.groupby('_ten')['Churn'].mean() * 100
colors = [C_CHURN if v > churn_rate else C_STAY for v in ten_c.values]

for i, (name, val) in enumerate(ten_c.items()):
    ax.plot(val, i, 'o', markersize=13, color=colors[i], zorder=3)
    ax.hlines(y=i, xmin=0, xmax=val, color=colors[i], linewidth=2.5, alpha=0.6)
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')

ax.axvline(churn_rate, color='#888', linestyle='--', linewidth=1.5, alpha=0.6)
ax.set_yticks(range(len(ten_c)))
ax.set_yticklabels(ten_c.index, fontsize=10)
ax.set_xlabel('Churn Rate (%)', fontsize=11)
ax.set_title('Churn by Tenure', fontsize=13, fontweight='bold', pad=10)
ax.grid(axis='x', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
df_clean.drop(columns=['_ten'], inplace=True, errors='ignore')


# ══════════════════════════════════════════════════════════════
# DIAGRAM — Referrals → Churn (Bar chart)
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5), facecolor=C_BG)
ax.set_facecolor(C_BG)

ref_c = df_clean.groupby('ReferralsMadeBySubscriber')['Churn'].mean() * 100
ref_c = ref_c.head(6)
colors = [C_CHURN if v > churn_rate else '#4A90D9' for v in ref_c.values]

bars = ax.bar(ref_c.index.astype(str), ref_c.values, color=colors,
              edgecolor='white', width=0.6)
for bar, val in zip(bars, ref_c.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

ax.axhline(churn_rate, color='#888', linestyle='--', linewidth=1.5, alpha=0.6,
           label=f'Avg {churn_rate:.1f}%')
ax.set_xlabel('Referrals Made', fontsize=11)
ax.set_ylabel('Churn Rate (%)', fontsize=11)
ax.set_title('Churn by Referrals Made', fontsize=13, fontweight='bold', pad=10)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()


# ══════════════════════════════════════════════════════════════
# DIAGRAM — Churn Rate by Overage Level (Bar chart)
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5), facecolor=C_BG)
ax.set_facecolor(C_BG)

df_clean['_ov'] = pd.cut(df_clean['OverageMinutes'],
    bins=[-1, 0, 50, 150, 500, 99999],
    labels=['Zero', '1-50', '51-150', '151-500', '500+'])
ov_c = df_clean.groupby('_ov')['Churn'].mean() * 100
colors = ['#E53E3E' if v > churn_rate else '#3182CE' for v in ov_c.values]

bars = ax.bar(ov_c.index.astype(str), ov_c.values, color=colors,
              edgecolor='white', width=0.6)
for bar, val in zip(bars, ov_c.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

ax.axhline(churn_rate, color='#888', linestyle='--', linewidth=1.5, alpha=0.6,
           label=f'Avg {churn_rate:.1f}%')
ax.set_xlabel('Overage Minutes', fontsize=11)
ax.set_ylabel('Churn %', fontsize=11)
ax.set_title('Churn Rate by Overage Level', fontsize=13, fontweight='bold', pad=10)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
df_clean.drop(columns=['_ov'], inplace=True, errors='ignore')


# ══════════════════════════════════════════════════════════════
# DIAGRAM — Retention & Loyalty: Referrals (Extended to 5)
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(8, 5), facecolor=C_BG)
ax.set_facecolor(C_BG)

ref_c2 = df_clean.groupby('ReferralsMadeBySubscriber')['Churn'].mean() * 100
ref_c2 = ref_c2.head(6)  # shows 0,1,2,3,4,5
colors = [C_CHURN if v > churn_rate else '#4A90D9' for v in ref_c2.values]

bars = ax.bar(ref_c2.index.astype(str), ref_c2.values, color=colors,
              edgecolor='white', width=0.6)
for bar, val in zip(bars, ref_c2.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

ax.axhline(churn_rate, color='#888', linestyle='--', linewidth=1.5, alpha=0.6,
           label=f'Avg {churn_rate:.1f}%')
ax.set_xlabel('Referrals Made', fontsize=11)
ax.set_ylabel('Churn %', fontsize=11)
ax.set_title('Churn by Referrals Made', fontsize=13, fontweight='bold', pad=10)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

print("✅ All 6 diagrams complete")

In [ ]:
# 4.2 — Churn rate by key categorical features
# For each category, compute: what % of customers in that group churned?
# Compare to the overall average (dashed line) to find high/low risk groups.

cat_eda = ['MadeCallToRetentionTeam', 'PrizmCode', 'Occupation',
           'MaritalStatus', 'Homeownership', 'HandsetRefurbished',
           'HandsetWebCapable', 'OwnsComputer']

overall_rate = df_clean['Churn'].mean() * 100

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(cat_eda):
    grp    = (df_clean.groupby(col)['Churn'].mean() * 100).sort_values(ascending=False)
    colors = ['#EF4444' if v > overall_rate else '#3B82F6' for v in grp.values]
    bars   = axes[i].bar(grp.index.astype(str), grp.values, color=colors, edgecolor='white')

    for bar, val in zip(bars, grp.values):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 0.3,
                     f'{val:.1f}%', ha='center', fontsize=9, fontweight='bold')

    axes[i].axhline(overall_rate, color='gray', linestyle='--', linewidth=1.2, alpha=0.8,
                    label=f'Avg {overall_rate:.1f}%')
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].set_ylim(0, grp.max() * 1.35)
    axes[i].tick_params(axis='x', rotation=20)
    axes[i].legend(fontsize=8)

plt.suptitle('Churn Rate by Categorical Feature  (red above average | blue below average)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

**Business insights:**
- **`MadeCallToRetentionTeam = Yes`** → dramatically higher churn — these customers are already actively leaving
- **`HandsetRefurbished = Yes`** → higher churn — satisfaction with refurbished phones is lower
- **`HandsetWebCapable = No`** → higher churn — older/less capable phones predict dissatisfaction
- **`OwnsComputer = Yes`** → slightly lower churn — tech-savvy customers may value connectivity more

In [ ]:
# 4.3 — Numeric feature distributions: churners vs non-churners
# For each key numeric feature, overlay two histograms:
# blue = non-churners, red = churners
# If the distributions overlap a lot → weak predictor
# If they separate clearly → strong predictor

num_eda = [
    'MonthlyRevenue',
    'MonthlyMinutes',
    'MonthsInService',
    'CurrentEquipmentDays',
    'CustomerCareCalls',
    'OverageMinutes',
    'TotalRecurringCharge'
]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, col in enumerate(num_eda):

    # cap extreme outliers for readability
    cap = df_clean[col].quantile(0.99)

    # split data by churn class
    d0 = df_clean.loc[df_clean['Churn'] == 0, col].dropna().clip(upper=cap)
    d1 = df_clean.loc[df_clean['Churn'] == 1, col].dropna().clip(upper=cap)

    # non-churners histogram
    axes[i].hist(
        d0,
        bins=40,
        alpha=0.55,
        color='#3B82F6',
        label='No Churn',
        density=True
    )

    # churners histogram
    axes[i].hist(
        d1,
        bins=40,
        alpha=0.55,
        color='#EF4444',
        label='Churn',
        density=True
    )

    # mean lines
    axes[i].axvline(
        d0.mean(),
        color='#1D4ED8',
        linestyle='--',
        linewidth=1.5
    )

    axes[i].axvline(
        d1.mean(),
        color='#B91C1C',
        linestyle='--',
        linewidth=1.5
    )

    # titles and labels
    axes[i].set_title(
        col,
        fontsize=11,
        fontweight='bold'
    )

    axes[i].set_xlabel('Value (capped at 99th pct)')
    axes[i].set_ylabel('Density')

    # legend only once
    if i == 0:
        axes[i].legend(fontsize=9)

# remove unused subplot completely
for j in range(len(num_eda), len(axes)):
    fig.delaxes(axes[j])

# overall title
plt.suptitle(
    'Numeric Feature Distributions: Churners vs Non-Churners\n(dashed = group mean)',
    fontsize=14,
    fontweight='bold',
    y=1.02
)

plt.tight_layout()
plt.show()

**Key findings:**
- **`CurrentEquipmentDays`** — churners have much higher values → old devices drive churn
- **`RetentionCalls`** — churners non-zero median, non-churners at zero → strong signal
- **`CustomerCareCalls`** — churners make more calls → frustration signal
- **`MonthsInService`** — non-churners tend to have longer tenure → loyalty effect
- **`MonthlyRevenue`** — slight rightward shift for churners → price sensitivity

In [ ]:
# 4.4 — Correlation heatmap
# Shows linear relationships between all numeric features.
# High correlation with Churn = individually predictive.
# High correlation between two features = potential redundancy.

num_cols_all = df_clean.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix  = df_clean[num_cols_all].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))   # hide upper triangle
fig, ax = plt.subplots(figsize=(18, 14))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.2, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with target
print("Top 12 features correlated with Churn:\n")
churn_corr = corr_matrix['Churn'].drop('Churn').abs().sort_values(ascending=False).head(12)
for feat, val in churn_corr.items():
    direction = '+' if corr_matrix['Churn'][feat] > 0 else '−'
    bar = '█' * int(val * 50)
    print(f"  {feat:<30}  r={direction}{val:.3f}  {bar}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# FEATURE-TO-FEATURE CORRELATION — Find Redundant Pairs
# ══════════════════════════════════════════════════════════════
# Two features highly correlated with EACH OTHER (|r| > 0.7)
# carry nearly the same information. Keeping both wastes a
# feature slot and can confuse linear models.
# Tree models handle it better but still benefit from cleanup.
# ══════════════════════════════════════════════════════════════

# Step 1: Find all pairs with |correlation| > 0.7
THRESHOLD = 0.7

corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        col_i = corr_matrix.columns[i]
        col_j = corr_matrix.columns[j]
        r = corr_matrix.iloc[i, j]
        if abs(r) > THRESHOLD and col_i != 'Churn' and col_j != 'Churn':
            corr_pairs.append({
                'Feature 1': col_i,
                'Feature 2': col_j,
                'Correlation': round(r, 3),
                '|r|': round(abs(r), 3)
            })

corr_pairs_df = pd.DataFrame(corr_pairs).sort_values('|r|', ascending=False).reset_index(drop=True)

print(f"Feature pairs with |correlation| > {THRESHOLD}:\n")
if len(corr_pairs_df) == 0:
    print("  ✅ No highly correlated feature pairs found.")
else:
    for _, row in corr_pairs_df.iterrows():
        r = row['Correlation']
        direction = "↑↑" if r > 0 else "↑↓"
        bar = '█' * int(row['|r|'] * 20)

        # Which one correlates more with Churn? That one is more useful
        corr_1 = abs(corr_matrix.loc[row['Feature 1'], 'Churn'])
        corr_2 = abs(corr_matrix.loc[row['Feature 2'], 'Churn'])

        if corr_1 >= corr_2:
            keep = row['Feature 1']
            drop_candidate = row['Feature 2']
        else:
            keep = row['Feature 2']
            drop_candidate = row['Feature 1']

        print(f"  {row['Feature 1']:<28} ←→ {row['Feature 2']:<28}  r={r:+.3f}  {bar}")
        print(f"    → Keep '{keep}' (stronger churn signal), consider dropping '{drop_candidate}'")
        print()

# Step 2: Visualise only the highly correlated pairs
if len(corr_pairs_df) > 0:
    # Build a focused heatmap with only the involved features
    involved = list(set(
        corr_pairs_df['Feature 1'].tolist() + corr_pairs_df['Feature 2'].tolist()
    ))
    involved_sorted = sorted(involved)

    focused_corr = corr_matrix.loc[involved_sorted, involved_sorted]
    mask_focused = np.triu(np.ones_like(focused_corr, dtype=bool))

    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(focused_corr, mask=mask_focused, annot=True, fmt='.2f',
                cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                linewidths=1, linecolor='white', ax=ax,
                annot_kws={'fontsize': 10, 'fontweight': 'bold'})
    ax.set_title(f'Focused Correlation: Features with |r| > {THRESHOLD}\n'
                 f'({len(corr_pairs_df)} redundant pairs found)',
                 fontsize=13, fontweight='bold', pad=12)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # Step 3: Summary table
    print(f"\n{'='*65}")
    print(f"  SUMMARY: {len(corr_pairs_df)} REDUNDANT FEATURE PAIRS FOUND")
    print(f"{'='*65}")
    print(f"\n  Candidates to DROP (less correlated with Churn):\n")

    drop_candidates = set()
    for _, row in corr_pairs_df.iterrows():
        corr_1 = abs(corr_matrix.loc[row['Feature 1'], 'Churn'])
        corr_2 = abs(corr_matrix.loc[row['Feature 2'], 'Churn'])
        if corr_1 >= corr_2:
            drop_candidates.add(row['Feature 2'])
        else:
            drop_candidates.add(row['Feature 1'])

    for feat in drop_candidates:
        partner_rows = corr_pairs_df[
            (corr_pairs_df['Feature 1'] == feat) | (corr_pairs_df['Feature 2'] == feat)
        ]
        partners = []
        for _, pr in partner_rows.iterrows():
            other = pr['Feature 2'] if pr['Feature 1'] == feat else pr['Feature 1']
            partners.append(f"{other} (r={pr['Correlation']:+.3f})")
        print(f"    🗑️  {feat}")
        print(f"        Redundant with: {', '.join(partners)}")
        print()

    print(f"  ACTION: You can safely remove these {len(drop_candidates)} features")
    print(f"  without losing information. The partner feature carries")
    print(f"  the same signal AND is more useful for churn prediction.")
else:
    print("\n✅ No redundant pairs — all features carry unique information.")

## Correlation Matrix Summary

The dataset shows several groups of strongly correlated features.

### Main Findings

* Usage features such as `MonthlyMinutes`, `ReceivedCalls`, `OutboundCalls`, `InboundCalls`, `PeakCallsInOut`, and `OffPeakCallsInOut` are highly correlated. They represent similar customer calling behavior.

* Revenue features such as `MonthlyRevenue`, `TotalRecurringCharge`, and `OverageMinutes` are positively correlated with usage. Higher usage usually means higher charges.

* Service quality features such as `DroppedCalls`, `BlockedCalls`, and `UnansweredCalls` are correlated, indicating network experience patterns.

* Household features `AgeHH1` and `AgeHH2` are strongly correlated.

* `UniqueSubs` and `ActiveSubs` are also strongly related.

### What This Means

There is feature redundancy in the dataset. Several variables carry overlapping information.

### Decisions for Modeling

* For tree-based models like LightGBM, XGBoost, or Random Forest, keep all features initially.

* For linear models like Logistic Regression, reduce highly correlated variables or apply regularization.

* Run feature importance after training to remove weak or duplicate variables.

* Consider creating grouped features such as total usage, billing score, or service quality score.

### Final Conclusion

The dataset is strong and informative, but feature selection will likely improve model efficiency and interpretability.


**Takeaways:**
- No single feature has strong correlation with churn (all < ~0.15) — this is normal
- Churn is **multi-factorial**: many weak signals combine into a strong prediction
- Tree-based models (Random Forest, LightGBM) are ideal here — they combine weak signals well
- `PeakCallsInOut` and `OffPeakCallsInOut` are highly correlated with each other — expected (complementary)

In [ ]:
# 4.5 — Boxplots: top numeric features vs churn
# Boxplots clearly show median, spread and outliers per group
# A visible difference in medians = predictive feature

top_num = ['CurrentEquipmentDays', 'MonthsInService', 'MonthlyRevenue',
           'CustomerCareCalls',
           'TotalRecurringCharge']

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(top_num):
    cap    = df_clean[col].quantile(0.97)
    subset = df_clean[[col, 'Churn']].copy()
    subset[col] = subset[col].clip(upper=cap)

    sns.boxplot(data=subset, x='Churn', y=col,
                palette={'0': '#3B82F6', '1': '#EF4444'},
                width=0.5, fliersize=2, ax=axes[i])
    axes[i].set_xticklabels(['No Churn (0)', 'Churn (1)'])
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Top Numeric Features vs Churn — Boxplot Comparison',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("How to read a boxplot:")
print("  Middle line  = median  |  Box = 25th–75th percentile")
print("  Whiskers     = min/max (excl. outliers)  |  Dots = outliers")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")


def detect_outliers_chart(df, target_col=None):

    num_cols = df.select_dtypes(include=np.number).columns.tolist()

    if target_col in num_cols:
        num_cols.remove(target_col)

    outlier_data = []

    for col in num_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        count = ((df[col] < lower) | (df[col] > upper)).sum()
        pct = round((count / len(df)) * 100, 2)

        outlier_data.append([col, count, pct])

    outlier_df = pd.DataFrame(
        outlier_data,
        columns=["Column", "Outlier Count", "Outlier %"]
    ).sort_values(by="Outlier Count", ascending=False)

    plt.figure(figsize=(12, 8))

    colors = [
        "#EF4444" if x > 10 else "#F59E0B" if x > 5 else "#3B82F6"
        for x in outlier_df["Outlier %"]
    ]

    bars = plt.barh(
        outlier_df["Column"],
        outlier_df["Outlier Count"],
        color=colors,
        edgecolor="white"
    )

    for bar, count, pct in zip(
        bars,
        outlier_df["Outlier Count"],
        outlier_df["Outlier %"]
    ):
        plt.text(
            bar.get_width() + 5,
            bar.get_y() + bar.get_height()/2,
            f"{count} ({pct}%)",
            va="center",
            fontsize=10,
            fontweight="bold"
        )

    plt.title(
        "Outlier Count by Feature (IQR Method)",
        fontsize=15,
        fontweight="bold"
    )

    plt.xlabel("Number of Outliers")
    plt.ylabel("Features")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

    return outlier_df


# Run
outlier_df = detect_outliers_chart(df_clean, target_col="Churn")

In [ ]:
# ── Interpretation of Outlier Analysis ────────────────────────
print("OUTLIER ANALYSIS — INTERPRETATION")
print("=" * 55)
print()
print("The IQR method flags many values as 'outliers' because")
print("telecom usage data is naturally RIGHT-SKEWED:")
print("  • Most customers make few calls → values cluster near 0")
print("  • Some customers are heavy users → long right tail")
print("  • The long tail is REAL behavior, not data errors")
print()
print("DECISION:")
print("  • Columns with >10% 'outliers' (red bars):")
print("    → NOT removed. These are valid skewed distributions.")
print("    → Tree models (LightGBM, XGBoost) handle skew natively.")
print()
print("  • We only remove values that are PHYSICALLY IMPOSSIBLE:")
print("    → Negative call counts (if any)")
print("    → Revenue = 999999 (system error)")
print("    → Age < 0 or Age > 120")
print()

# Check for actually impossible values
impossible = {}
if (df_clean['MonthlyRevenue'] < 0).any():
    impossible['MonthlyRevenue < 0'] = (df_clean['MonthlyRevenue'] < 0).sum()
if (df_clean['MonthlyMinutes'] < 0).any():
    impossible['MonthlyMinutes < 0'] = (df_clean['MonthlyMinutes'] < 0).sum()
if (df_clean['CurrentEquipmentDays'] < 0).any():
    impossible['CurrentEquipmentDays < 0'] = (df_clean['CurrentEquipmentDays'] < 0).sum()
if (df_clean['MonthsInService'] < 0).any():
    impossible['MonthsInService < 0'] = (df_clean['MonthsInService'] < 0).sum()

if impossible:
    print("⚠️  IMPOSSIBLE values found (should be removed):")
    for k, v in impossible.items():
        print(f"    {k}: {v} rows")
else:
    print("✅ No impossible values found.")
    print("   All 'outliers' are valid extreme values.")
    print("   No rows need to be removed.")
print()
print("WHY WE KEEP THEM:")
print("  1. Our model is LightGBM — tree-based, not affected by outliers")
print("  2. Extreme values ARE the churn signal (e.g. 15+ care calls)")
print("  3. Removing 25% of PercChangeRevenues would destroy the dataset")
print("  4. IQR method is designed for normal distributions,")
print("     not skewed telecom data")

In [ ]:
df_clean.describe()

---
## Section 5 — Separating Features and Target

### 📖 What and Why

Every sklearn model needs:
- **`X`** — feature matrix (all columns except target)
- **`y`** — target vector (just the `Churn` column)

We also classify features into **numeric** and **categorical** groups
because they need different preprocessing.

In [ ]:
# Separate features (X) and target (y)
X = df_clean.drop(columns=['Churn'])
y = df_clean['Churn']

print(f"Feature matrix X : {X.shape[0]:,} rows  ×  {X.shape[1]} features")
print(f"Target vector  y : {y.shape[0]:,} entries  (0 = No Churn, 1 = Churn)")

# Classify by dtype for the preprocessing pipeline
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object','string']).columns.tolist()

print(f"\nNumeric features     ({len(numeric_cols)}):")
print(f"  {numeric_cols}")
print(f"\nCategorical features ({len(categorical_cols)}):")
print(f"  {categorical_cols}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Section 5.5 — Feature Engineering
# ══════════════════════════════════════════════════════════════
# Create new features that capture RELATIONSHIPS between existing columns.
# A single raw column like MonthlyRevenue means little on its own.
# But MonthlyRevenue RELATIVE TO MonthsInService tells you if a
# customer is overpaying for their tenure — that's a churn signal.
# ══════════════════════════════════════════════════════════════

def add_features(df):
    """Add ratio and flag features. Used for both training and holdout data."""
    df = df.copy()

    # ── Usage efficiency ratios ───────────────────────────────────
    # Revenue per minute: are they paying a lot for little usage?
    df['RevenuePerMinute'] = df['MonthlyRevenue'] / (df['MonthlyMinutes'] + 1)

    # Overage rate: how often do they exceed their plan?
    df['OverageRate'] = df['OverageMinutes'] / (df['MonthlyMinutes'] + 1)

    # ── Frustration signals ──────────────────────────────────────
    # Care calls per month of tenure: new customers calling a lot = bad sign
    df['CareCallsPerMonth'] = df['CustomerCareCalls'] / (df['MonthsInService'] + 1)

    # Drop rate: quality of service the customer actually experiences
    df['DropRate'] = df['DroppedCalls'] / (df['ReceivedCalls'] + df['OutboundCalls'] + 1)

    # ── Price sensitivity ─────────────────────────────────────────
    # Charge relative to tenure: long-tenure customers paying a lot may leave
    df['ChargePerTenureMonth'] = df['TotalRecurringCharge'] / (df['MonthsInService'] + 1)

    # ── Engagement change ─────────────────────────────────────────
    # Combined usage change: are both minutes AND revenue declining?
    df['UsageDecline'] = (
        (df['PercChangeMinutes'] < 0) & (df['PercChangeRevenues'] < 0)
    ).astype(int)


    # Already called retention but got no offer accepted
    if 'RetentionCalls' in df.columns and 'RetentionOffersAccepted' in df.columns:
        df['RetentionFailed'] = (
            (df['RetentionCalls'] > 0) & (df['RetentionOffersAccepted'] == 0)
        ).astype(int)
    return df


X = add_features(X)

# ── Update column lists ──────────────────────────────────────
new_numeric = ['RevenuePerMinute', 'OverageRate', 'CareCallsPerMonth',
               'DropRate', 'ChargePerTenureMonth']
new_binary  = ['UsageDecline']

# Add to numeric_cols (binary flags work fine as numeric for trees)
for col in new_numeric + new_binary:
    if col in X.columns and col not in numeric_cols:
        numeric_cols.append(col)

if 'RetentionFailed' in X.columns and 'RetentionFailed' not in numeric_cols:
    numeric_cols.append('RetentionFailed')

print(f"✅ Feature engineering complete!")
print(f"   New features added: {len(new_numeric) + len(new_binary) + (1 if 'RetentionFailed' in X.columns else 0)}")
print(f"   Total features now: {X.shape[1]}")
print(f"   Numeric: {len(numeric_cols)}  |  Categorical: {len(categorical_cols)}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Remove Redundant Correlated Features (Conservative)
# ══════════════════════════════════════════════════════════════
# Only remove features that are TRULY redundant — where the
# partner carries the same or better signal.
# Keep features that have unique behavioral meaning for churn.
#
# Rules:
#   1. Never remove ALL features from a behavioral group
#   2. Keep features with direct business meaning for churn
#   3. Remove sums/aggregates when components are kept
#   4. Remove duplicates, not unique signals
# ══════════════════════════════════════════════════════════════

cols_to_drop = [
    # ── True duplicates / sums ────────────────────────────────
    'DroppedBlockedCalls',      # SUM of DroppedCalls + BlockedCalls — keeping both components
    'Handsets',                 # r=0.89 with HandsetModels — nearly identical, keep Models

    # ── Redundant call breakdowns ─────────────────────────────
    # MonthlyMinutes already captures total call volume.
    # We keep ONE representative from each call group.
    'PeakCallsInOut',           # r=0.77 with MonthlyMinutes — same usage signal
    'OffPeakCallsInOut',        # r=0.77 with MonthlyMinutes — same usage signal
    'OutboundCalls',            # r=0.74 with OffPeak + r=0.72 with InboundCalls

    # ── Redundant account features ────────────────────────────
    'ActiveSubs',               # r=0.78 with UniqueSubs — nearly identical
    'RetentionOffersAccepted',  # r=0.73 with RetentionCalls — if you got offers, you got calls
]

# ── EXPLICITLY KEPT (and why) ─────────────────────────────────
# MonthlyRevenue      → KEEP: direct price sensitivity signal, business KPI
# BlockedCalls        → KEEP: network quality signal (DroppedBlockedCalls removed instead)
# DroppedCalls        → KEEP: different frustration signal than BlockedCalls
# UnansweredCalls     → KEEP: unique frustration signal, not captured by other features
# ReceivedCalls       → KEEP: represents customer engagement (people calling them)
# InboundCalls        → KEEP: internal network activity, different from ReceivedCalls
# RetentionCalls      → KEEP: strongest retention signal (partner removed instead)

# Only drop columns that exist
existing_to_drop = [c for c in cols_to_drop if c in X.columns]
X = X.drop(columns=existing_to_drop)

# Update column lists
numeric_cols     = [c for c in numeric_cols if c not in existing_to_drop]
categorical_cols = [c for c in categorical_cols if c not in existing_to_drop]

print(f"Removed {len(existing_to_drop)} truly redundant features:")
for col in existing_to_drop:
    print(f"  🗑️  {col}")

print(f"\nKept (important behavioral signals):")
kept_notes = {
    'MonthlyRevenue':   'price sensitivity, business KPI',
    'BlockedCalls':     'network quality frustration',
    'DroppedCalls':     'call quality frustration',
    'UnansweredCalls':  'customer frustration signal',
    'ReceivedCalls':    'customer engagement level',
    'InboundCalls':     'internal network activity',
    'RetentionCalls':   'strongest retention signal',
}
for feat, reason in kept_notes.items():
    if feat in X.columns:
        print(f"  ✅  {feat:<28} → {reason}")

print(f"\nRemaining features: {X.shape[1]}")
print(f"  Numeric:     {len(numeric_cols)}")
print(f"  Categorical: {len(categorical_cols)}")

**Why `OrdinalEncoder` instead of `OneHotEncoder`?**

For tree-based models (Random Forest, XGBoost, LightGBM) — our top performers —
`OrdinalEncoder` works perfectly. Trees split on numeric thresholds; they don't need
dummy variables. `OneHotEncoder` would create 30+ extra columns with minimal benefit,
and would crash on unseen category values in the holdout set.

---
## Section 6 — Building the Preprocessing Pipeline

### 📖 What and Why

A scikit-learn **Pipeline** chains preprocessing and modelling into one object.
This is not just convenient — it is **essential for correctness**:

| Risk | How Pipeline solves it |
|------|----------------------|
| Data leakage | Imputers/scalers fit on training data only, then transform val/test |
| Reproducibility | One object = entire workflow |
| Deployment safety | Ship one pipeline, not scattered loose transformers |

**Our pipeline has two branches:**
- **Numeric:** Median imputation → StandardScaler
- **Categorical:** Constant imputation → OrdinalEncoder

>  **Data leakage warning:** If you called `scaler.fit(X_all) z=x−mean/std` before splitting,
> the scaler would "know" validation statistics. This inflates your metrics artificially.
> A Pipeline fitted on `X_train` prevents this completely.

In [ ]:
# (Numeric) pipeline: fill missing with median, then z-score normalise
# Median is robust to outliers (better than mean for skewed distributions)
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())    # z-score: (x - mean) / std
])

# (Categorical) pipeline: fill missing with constant string, then ordinal encode
# 'missing' is treated as its own category by the model
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OrdinalEncoder(
        handle_unknown='use_encoded_value',   # unseen values → -1 (safe for holdout)
        unknown_value=-1
    ))
])

# ColumnTransformer applies the right pipeline to the right column groups
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer,     numeric_cols),       # ~36 numeric features
    ('cat', categorical_transformer, categorical_cols)    # ~18 categorical features
])

print("Preprocessing pipeline summary:")
print(f"  Numeric  branch  : {len(numeric_cols)} features → median impute + StandardScaler")
print(f"  Categorical branch: {len(categorical_cols)} features → constant impute + OrdinalEncoder")
print(f"  Total output features: {len(numeric_cols) + len(categorical_cols)}")
print()
print("✅ Preprocessor defined.")
print("   It will be fitted ONLY on X_train inside each Pipeline — never on validation/holdout.")

---
## Section 7 — Train / Validation Split

### 📖 What and Why

We split our cleaned training data into:
- **`X_train / y_train`** (80%) → fit models and preprocessing
- **`X_val / y_val`** (20%) → evaluate models

`stratify=y` ensures both halves preserve the 28.8% churn rate.
Without stratification, random chance could put most churners in one half → misleading metrics.

> **`df_holdout` is not touched here.** It stays completely locked away.

In [ ]:
# Stratified 80/20 split of training data only
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,       # 80% train, 20% validation
    random_state=42,      # fixed seed for full reproducibility
    stratify=y            # preserve churn rate in both halves
)

print("Split summary:")
print(f"  X_train : {X_train.shape[0]:>6,} rows  |  churn rate: {y_train.mean()*100:.1f}%")
print(f"  X_val   : {X_val.shape[0]:>6,} rows  |  churn rate: {y_val.mean()*100:.1f}%")
print()
print(" Both halves have ~28.8% churn — stratification confirmed.")
print(" df_holdout remains completely untouched until Section 12.")

---
## Section 8 — Baseline & Model Comparison

### 📖 What and Why

We train **7 models** (including a baseline) and compare them on the same validation set.

| Model | Why include it |
|-------|---------------|
| Logistic Regression | Linear, fast, interpretable — good sanity check |
| Random Forest | Ensemble of 300 trees — robust and strong |
| XGBoost | Gradient boosting — gold standard for tabular data |
| LightGBM | Faster gradient boosting — often matches or beats XGBoost |

All models use `class_weight='balanced'` — this upweights churner examples so the model
pays more attention to the minority class during training.

---

### 📏 Why these metrics matter for churn prediction

| Metric | What it measures | For churn |
|--------|-----------------|-----------|
| **Accuracy** | (TP+TN) / all | ⚠️ Misleading with imbalance |
| **Precision** | TP / (TP+FP) | Of customers we flag, how many truly churn? |
| **Recall** | TP / (TP+FN) | Of real churners, how many do we catch? ← **Most important** |
| **F1-score** | Harmonic mean | Balance of precision & recall |
| **ROC-AUC** | Area under ROC | Overall discrimination — threshold-free ← **Very important** |

> **Key insight:** Missing a churner (False Negative) costs a full customer.
> A false alarm (False Positive) costs one small retention offer.
> The cost asymmetry means **Recall and AUC are the most important metrics here.**

In [ ]:
# Helper: fit a pipeline on training data and evaluate on validation data
def evaluate_pipeline(pipeline, X_tr, y_tr, X_v, y_v, threshold=0.5, name=""):
    """Fit pipeline on training data, evaluate on validation data."""
    pipeline.fit(X_tr, y_tr)

    # predict_proba gives probabilities — needed for AUC and custom threshold
    y_proba = pipeline.predict_proba(X_v)[:, 1]   # P(churn)
    y_pred  = (y_proba >= threshold).astype(int)

    return {
        'name'      : name,
        'pipeline'  : pipeline,
        'accuracy'  : round(accuracy_score(y_v, y_pred), 4),
        'precision' : round(precision_score(y_v, y_pred, zero_division=0), 4),
        'recall'    : round(recall_score(y_v, y_pred, zero_division=0), 4),
        'f1'        : round(f1_score(y_v, y_pred, zero_division=0), 4),
        'roc_auc'   : round(roc_auc_score(y_v, y_proba), 4),
        'y_pred'    : y_pred,
        'y_proba'   : y_proba,
    }

In [ ]:
# Define all 4 model pipelines
# scale_pos_weight tells XGBoost how much more to weight the minority class
scale_pos_weight = (y_train==0).sum() / (y_train==1).sum()

models = {
#pre = imputer + scale

    'Logistic Regression' : Pipeline([('pre', preprocessor),
        ('clf', LogisticRegression(class_weight='balanced', C=1.0,
                                   max_iter=1000, random_state=42))]),


    'Random Forest' : Pipeline([('pre', preprocessor),
        ('clf', RandomForestClassifier(class_weight='balanced', n_estimators=300,
                                       max_depth=12, n_jobs=-1, random_state=42))]),

    'XGBoost' : Pipeline([('pre', preprocessor),
        ('clf', XGBClassifier(scale_pos_weight=scale_pos_weight,
                              n_estimators=300, max_depth=6, learning_rate=0.05,
                              subsample=0.8, colsample_bytree=0.8,
                              random_state=42, eval_metric='logloss', verbosity=0))]),

    'LightGBM' : Pipeline([('pre', preprocessor),
        ('clf', LGBMClassifier(class_weight='balanced', n_estimators=300,
                               max_depth=6, learning_rate=0.05,
                               random_state=42, verbose=-1))]),

}

print(f"✅ {len(models)} pipelines defined. Ready to train.")

In [ ]:
# Train all models and collect metrics
# Each model is trained on X_train and evaluated on X_val
# Preprocessing (imputer + scaler) is fitted on X_train only

results = {}

print(f"  {'Model':<22} {'   Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'AUC':>7}")
print("  " + "─"*57)

for name, pipeline in models.items():
    print(f"  Training {name:<22}", end="", flush=True)
    r = evaluate_pipeline(pipeline, X_train, y_train, X_val, y_val, name=name)
    results[name] = r
    print(f"{r['accuracy']:>6.4f} {r['precision']:>6.4f} {r['recall']:>6.4f} "
          f"{r['f1']:>6.4f} {r['roc_auc']:>7.4f}")

print("\n✅ All models trained and evaluated on the validation set.")

## Feature Selection Impact Analysis

To evaluate the effect of the correlation-based feature selection, model performance was compared with and without the redundant features. *(Numbers recorded from an earlier experiment run, before the engineered ratio features were added.)*

### Results before feature selection

| Model | Accuracy | Precision | Recall | F1 Score | AUC |
|------|---------:|----------:|-------:|---------:|----:|
| Logistic Regression | 0.5919 | 0.3677 | 0.5785 | 0.4496 | 0.6150 |
| Random Forest | 0.6393 | 0.4061 | 0.5449 | 0.4654 | 0.6583 |
| XGBoost | 0.6425 | 0.4166 | 0.6010 | 0.4921 | 0.6770 |
| LightGBM | 0.6356 | 0.4126 | 0.6251 | 0.4971 | 0.6799 |

---

### Results after feature selection

| Model | Accuracy | Precision | Recall | F1 Score | AUC |
|------|---------:|----------:|-------:|---------:|----:|
| Logistic Regression | 0.5884 | 0.3626 | 0.5649 | 0.4417 | 0.6108 |
| Random Forest | 0.6311 | 0.3982 | 0.5479 | 0.4612 | 0.6593 |
| XGBoost | 0.6404 | 0.4145 | 0.6016 | 0.4908 | 0.6786 |
| LightGBM | 0.6286 | 0.4048 | 0.6139 | 0.4878 | 0.6786 |

---

### Interpretation

The differences between both experiments were very small across all models. This indicates that the removed features had limited predictive value and contributed little useful information.

### Key Findings

- **Random Forest** improved slightly after feature selection.
- **XGBoost** also improved slightly and remained one of the top models.
- **LightGBM** decreased only marginally (0.0013 AUC), which is statistically negligible.
- **Logistic Regression** declined slightly, suggesting linear models may benefit from keeping some weaker variables.

### Conclusion

Feature Selection was successful because it reduced dimensionality without harming predictive performance.  
This leads to:

- Faster training time  
- Lower model complexity  
- Reduced overfitting risk  
- Easier interpretation  

### Final Decision

The reduced feature set will be retained for the remaining modelling stages, including:

- Hyperparameter Tuning  
- Threshold Optimization  
- Final Holdout Evaluation  

Since **LightGBM** and **XGBoost** achieved the highest AUC scores, they remain the strongest candidate models for final deployment.

In [ ]:
# Formatted comparison table sorted by ROC-AUC
metrics = ['accuracy','precision','recall','f1','roc_auc']
comp_df = pd.DataFrame([
    {'Model': r['name'], **{m: r[m] for m in metrics}}
    for r in results.values()
]).sort_values('roc_auc', ascending=False).reset_index(drop=True)

def highlight_best(col):
    return ['background-color: #BBF7D0; font-weight: bold'
            if v == col.max() else '' for v in col]

print("Model Comparison Table (sorted by ROC-AUC):\n")
display(
    comp_df.style
    .apply(highlight_best, subset=metrics)
    .format({m: '{:.4f}' for m in metrics})
    .set_caption("🟢 Green = best value in each column")
)

In [ ]:
# ROC curves for all models
fig, ax = plt.subplots(figsize=(9, 7))
colors  = plt.cm.tab10(np.linspace(0, 1, len(results)))

for (name, r), color in zip(results.items(), colors):
    if name == 'Dummy (Majority)':
        continue
    fpr, tpr, _ = roc_curve(y_val, r['y_proba'])
    ax.plot(fpr, tpr, linewidth=2, color=color,
            label=f"{name}  (AUC = {r['roc_auc']:.3f})")

ax.plot([0,1],[0,1], 'k--', linewidth=1, alpha=0.5, label='Random classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curves — All Models on Validation Set', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("How to read ROC curves:")
print("  • Curve closer to TOP-LEFT corner = better model")
print("  • AUC = 1.0 → perfect  |  AUC = 0.5 → random guessing")
print("  • Dashed diagonal = a model that just guesses randomly")

In [ ]:
# Bar chart comparison of all metrics
comp_no_dummy = comp_df[comp_df['Model'] != 'Dummy (Majority)'].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 6))
x      = np.arange(len(comp_no_dummy))
width  = 0.15
mcolors = ['#3B82F6','#EF4444','#10B981','#F59E0B','#8B5CF6']
mlabels = ['Accuracy','Precision','Recall','F1-Score','ROC-AUC']

for i, (m, c, l) in enumerate(zip(metrics, mcolors, mlabels)):
    ax.bar(x + i*width, comp_no_dummy[m], width, label=l, color=c, alpha=0.85, edgecolor='white')

ax.set_xticks(x + width*2)
ax.set_xticklabels(comp_no_dummy['Model'], rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.1)
ax.set_title('Full Metric Comparison Across All Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.axhline(0.8, color='gray', linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

**How to interpret the comparison:**

A naive model that always predicts "No Churn" reaches ~71% **accuracy** but **0% recall** — it never finds a churner.
This is why accuracy alone is misleading for imbalanced problems, and why ROC-AUC and Recall are the main metrics here.

Ranking by AUC: **XGBoost ≈ LightGBM > Random Forest > Logistic Regression**

→ The two gradient-boosting models are taken forward to threshold optimisation and tuning.

---
## Section 9 — Handling Class Imbalance

### 📖 What and Why

We already used `class_weight='balanced'` in all models. Now we go further:

**Technique 1 — Threshold Optimisation:**
The default decision boundary is P(churn) ≥ 0.5. With class imbalance, models
are conservative about predicting "churn". Lowering the threshold catches more churners.
We find the optimal threshold on the **validation set only**.

**Technique 2 — SMOTE:**
Creates synthetic new churner examples by interpolating between real ones.
Applied **only to training data** — never validation or holdout.

>  **Critical rule:** If you apply SMOTE before splitting, synthetic points from training
> can contaminate the validation set, making recall appear artificially high.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9.1 — Business-Aware Threshold Optimisation
# Goal:
#   Choose a threshold that is not only good mathematically,
#   but also practical for a retention team.
#
# Rule:
#   Recall >= 60%
#   Flagged customers <= 45%
# ══════════════════════════════════════════════════════════════
from sklearn.metrics import classification_report
# Select best base model by AUC
best_base_name = comp_df.iloc[0]['Model']
print(f"Optimising threshold for: {best_base_name}")

# Probabilities from the best model on validation set
y_proba_best = results[best_base_name]['y_proba']

# Test thresholds from 0.10 to 0.90
thresholds = np.arange(0.10, 0.90, 0.01)
thr_results = []

for t in thresholds:
    y_t = (y_proba_best >= t).astype(int)

    flagged_pct = y_t.mean() * 100

    thr_results.append({
        'threshold'   : round(t, 2),
        'accuracy'    : round(accuracy_score(y_val, y_t), 4),
        'precision'   : round(precision_score(y_val, y_t, zero_division=0), 4),
        'recall'      : round(recall_score(y_val, y_t, zero_division=0), 4),
        'f1'          : round(f1_score(y_val, y_t, zero_division=0), 4),
        'flagged_pct' : round(flagged_pct, 1)
    })

thr_df = pd.DataFrame(thr_results)

# Default threshold row
row50 = thr_df[thr_df['threshold'] == 0.50].iloc[0]

# Business rule:
# Keep only thresholds that catch at least 60% of churners
# and flag at most 45% of all customers
viable = thr_df[
    (thr_df['recall'] >= 0.60) &
    (thr_df['flagged_pct'] <= 45)
]

if len(viable) > 0:
    # From practical thresholds, choose the one with best F1
    best_row = viable.loc[viable['f1'].idxmax()]
    selection_reason = "Best F1 among thresholds with Recall >= 60% and Flagged <= 45%"
else:
    # Fallback if no threshold satisfies the business rule
    best_row = thr_df.loc[thr_df['f1'].idxmax()]
    selection_reason = "Fallback: no threshold satisfied the business rule, so best F1 was selected"

optimal_threshold = best_row['threshold']

print("\nDefault threshold (0.50):")
print(f"  Accuracy     = {row50['accuracy']:.4f}")
print(f"  Precision    = {row50['precision']:.4f}")
print(f"  Recall       = {row50['recall']:.4f}")
print(f"  F1           = {row50['f1']:.4f}")
print(f"  Flagged %    = {row50['flagged_pct']:.1f}%")

print(f"\nSelected threshold ({optimal_threshold:.2f}):")
print(f"  Accuracy     = {best_row['accuracy']:.4f}")
print(f"  Precision    = {best_row['precision']:.4f}")
print(f"  Recall       = {best_row['recall']:.4f}")
print(f"  F1           = {best_row['f1']:.4f}")
print(f"  Flagged %    = {best_row['flagged_pct']:.1f}%")
print(f"  Reason       = {selection_reason}")

# Final predictions using selected threshold
y_pred_optimized = (y_proba_best >= optimal_threshold).astype(int)

print("\nFinal classification report using selected threshold:")
print(classification_report(y_val, y_pred_optimized, zero_division=0))

print("\nThreshold candidates that satisfy the business rule:")
print(
    viable.sort_values(by='f1', ascending=False)
          .head(10)
          .to_string(index=False)
)

In [ ]:
import time

# Get the original pipelines for XGBoost and LightGBM from the 'models' dictionary
xgb_pipeline = models['XGBoost']
lgbm_pipeline = models['LightGBM']

start = time.time()
xgb_pipeline.fit(X_train, y_train) # Fit the pipeline on X_train and y_train
xgb_time = time.time() - start

start = time.time()
lgbm_pipeline.fit(X_train, y_train) # Fit the pipeline on X_train and y_train
lgbm_time = time.time() - start

print(f"XGBoost:  {xgb_time:.2f}s")
print(f"LightGBM: {lgbm_time:.2f}s")
print(f"Speed-up: {xgb_time/lgbm_time:.1f}x")

In [ ]:
# Visualise threshold vs metrics
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thr_df['threshold'], thr_df['precision'], label='Precision', color='#3B82F6', lw=2)
ax.plot(thr_df['threshold'], thr_df['recall'],    label='Recall',    color='#EF4444', lw=2)
ax.plot(thr_df['threshold'], thr_df['f1'],        label='F1-Score',  color='#10B981', lw=2.5)
ax.axvline(optimal_threshold, color='#F59E0B', linestyle='--', lw=2,
           label=f'Optimal = {optimal_threshold}')
ax.axvline(0.5, color='gray', linestyle=':', lw=1.5, label='Default = 0.50')
ax.set_xlabel('Decision Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'Threshold Optimisation — {best_base_name}', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0.1, 0.9)
plt.tight_layout()
plt.show()

print(f"\nBusiness interpretation:")
print(f"  Lowering the threshold from 0.50 to {optimal_threshold} flags more customers as churners.")
print(f"  This means more retention calls (cost) but catches more real churners (savings).")
print(f"  The optimal threshold depends on the business cost of false positives vs false negatives.")

In [ ]:
# 9.2 — SMOTE: oversample minority class in training data only
# Step 1: Transform training data with the fitted preprocessor
# Step 2: Apply SMOTE to create synthetic churner examples
# Step 3: Train on balanced data WITH class_weight='balanced'
# Step 4: Evaluate on ORIGINAL (unmodified) validation set

pre_smote = ColumnTransformer(transformers=[
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                      ('sc', StandardScaler())]), numeric_cols),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                      ('enc', OrdinalEncoder(handle_unknown='use_encoded_value',
                                             unknown_value=-1))]), categorical_cols)
])

X_train_pp = pre_smote.fit_transform(X_train)   # fit on train, transform train
X_val_pp   = pre_smote.transform(X_val)          # transform val (NO fitting)

print(f"Class distribution BEFORE SMOTE: {dict(zip(*np.unique(y_train, return_counts=True)))}")
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_pp, y_train)
print(f"Class distribution AFTER  SMOTE: {dict(zip(*np.unique(y_train_sm, return_counts=True)))}")
print(f"Training size grew: {len(X_train_pp):,} → {len(X_train_sm):,} rows")

# ╔══════════════════════════════════════════════════════════════╗
# ║  CHANGE 1: Added class_weight='balanced'                    ║
# ║                                                              ║
# ║  WHY: SMOTE balances the training DATA, but the model's     ║
# ║  internal loss function still treats both classes equally.   ║
# ║  Without class_weight, the model learns that predicting     ║
# ║  "No Churn" is safe because the loss penalty is the same.   ║
# ║  class_weight='balanced' makes misclassifying a churner     ║
# ║  MORE expensive than misclassifying a non-churner.           ║
# ║  SMOTE + class_weight together = strongest imbalance fix.    ║
# ╚══════════════════════════════════════════════════════════════╝
lgbm_smote = LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    class_weight='balanced',    # ← ADDED: was missing, caused Recall=0.15
    random_state=42,
    verbose=-1
)
lgbm_smote.fit(X_train_sm, y_train_sm)

y_proba_smote = lgbm_smote.predict_proba(X_val_pp)[:, 1]

# ╔══════════════════════════════════════════════════════════════╗
# ║  CHANGE 2: Also evaluate at optimal_threshold               ║
# ║                                                              ║
# ║  WHY: Your Section 9.1 already found that 0.50 is not the   ║
# ║  best threshold. Evaluating SMOTE only at 0.50 makes the    ║
# ║  comparison unfair against your tuned baseline model.        ║
# ║  Showing both thresholds gives a fair comparison.            ║
# ╚══════════════════════════════════════════════════════════════╝
y_pred_smote_default  = (y_proba_smote >= 0.5).astype(int)
y_pred_smote_optimal  = (y_proba_smote >= optimal_threshold).astype(int)

print(f"\nLightGBM + SMOTE (threshold=0.50):")
print(f"  Accuracy  = {accuracy_score(y_val, y_pred_smote_default):.4f}")
print(f"  Precision = {precision_score(y_val, y_pred_smote_default, zero_division=0):.4f}")
print(f"  Recall    = {recall_score(y_val, y_pred_smote_default, zero_division=0):.4f}")
print(f"  F1        = {f1_score(y_val, y_pred_smote_default, zero_division=0):.4f}")
print(f"  ROC-AUC   = {roc_auc_score(y_val, y_proba_smote):.4f}")

print(f"\nLightGBM + SMOTE (threshold={optimal_threshold}):")
print(f"  Accuracy  = {accuracy_score(y_val, y_pred_smote_optimal):.4f}")
print(f"  Precision = {precision_score(y_val, y_pred_smote_optimal, zero_division=0):.4f}")
print(f"  Recall    = {recall_score(y_val, y_pred_smote_optimal, zero_division=0):.4f}")
print(f"  F1        = {f1_score(y_val, y_pred_smote_optimal, zero_division=0):.4f}")
print(f"  ROC-AUC   = {roc_auc_score(y_val, y_proba_smote):.4f}")

# ╔══════════════════════════════════════════════════════════════╗
# ║  CHANGE 3: Explicit comparison against baseline              ║
# ║                                                              ║
# ║  WHY: You need to document whether SMOTE actually helped.    ║
# ║  This prints a clear verdict so your notebook shows the      ║
# ║  decision logic, not just raw numbers.                       ║
# ╚══════════════════════════════════════════════════════════════╝
best_base_auc  = results[comp_df.iloc[0]['Model']]['roc_auc']
smote_auc      = round(roc_auc_score(y_val, y_proba_smote), 4)

print(f"\n{'='*55}")
print(f"  SMOTE COMPARISON")
print(f"{'='*55}")
print(f"  Baseline best model AUC : {best_base_auc:.4f}")
print(f"  SMOTE LightGBM AUC     : {smote_auc:.4f}")
print(f"  Difference              : {smote_auc - best_base_auc:+.4f}")

if smote_auc > best_base_auc:
    print(f"\n  ✅ SMOTE improved AUC by {smote_auc - best_base_auc:.4f}")
    print(f"     → Consider using SMOTE in the final pipeline")
else:
    print(f"\n  ⚠️  SMOTE did not improve AUC")
    print(f"     → Baseline with class_weight='balanced' is sufficient")
    print(f"     → Proceeding with baseline model for tuning")

## SMOTE Evaluation Results

SMOTE was applied to the **training data only** (29,068 non-churners / 11,769 churners → 29,068 / 29,068),
and LightGBM was evaluated on the original, untouched validation set.

**Conclusion:** SMOTE did **not** improve ROC-AUC compared to the best class-weighted baseline (see the comparison printed above).
The class imbalance (≈71:29) is only moderate and is already handled by `class_weight='balanced'`
plus a tuned decision threshold. SMOTE is therefore not used in the final pipeline.

**SMOTE findings:**
SMOTE can improve recall but often trades precision. Compare:
- Base model + `class_weight='balanced'` + default threshold
- Base model + optimised threshold
- SMOTE model

In practice for structured tabular data, **threshold optimisation usually outperforms SMOTE**
because SMOTE's synthetic samples may not perfectly reflect real customer patterns.

---
## Section 10 — Hyperparameter Tuning

### 📖 What and Why

Default hyperparameters are reasonable starting points but rarely optimal.
**Hyperparameters** are settings chosen before training — not learned from data.
Examples: number of trees, learning rate, tree depth, regularisation strength.

**RandomizedSearchCV** tests 80 random combinations and evaluates each using
5-fold cross-validation on training data only.

> **Why RandomizedSearch and not GridSearch?**
> GridSearch tests every combination → exponential cost.
> RandomizedSearch tests a random sample → much faster, nearly as effective.

> **Why cross-validation during tuning?**
> If we used the val set to tune, we'd overfit to that specific val split.
> CV uses only training data, keeping val as a genuine independent check.

In [ ]:
# Build a fresh LightGBM pipeline for tuning
lgbm_pipeline = Pipeline([
    ('pre', ColumnTransformer(transformers=[
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('sc',  StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                          ('enc', OrdinalEncoder(handle_unknown='use_encoded_value',
                                                 unknown_value=-1))]), categorical_cols)
    ])),
    ('clf', LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1))
])

# Search space — prefix 'clf__' tells sklearn to pass param to the classifier step
param_dist = {
    'clf__n_estimators'      : randint(100, 500),
    'clf__max_depth'         : randint(3, 9),
    'clf__learning_rate'     : uniform(0.01, 0.19),
    'clf__num_leaves'        : randint(20, 80),
    'clf__min_child_samples' : randint(10, 60),
    'clf__subsample'         : uniform(0.6, 0.4),
    'clf__colsample_bytree'  : uniform(0.5, 0.5),
    'clf__reg_alpha'         : uniform(0.0, 1.0),
    'clf__reg_lambda'        : uniform(0.0, 1.0),
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    lgbm_pipeline,
    param_distributions=param_dist,
    n_iter=80,            # 80 random combinations
    scoring='roc_auc',    # optimise for AUC
    cv=cv_strategy,       # 5-fold CV on training data only
    n_jobs=-1,
    random_state=42,
    verbose=1
)

print("Starting hyperparameter search (80 iterations × 5-fold CV = 400 fits)...")
print("This may take 10–20 minutes...\n")
search.fit(X_train, y_train)

print(f"\n✅ Search complete!")
print(f"   Best CV AUC  : {search.best_score_:.4f}")
print(f"   Best params  :")
for k, v in search.best_params_.items():
    print(f"     {k}: {v}")

In [ ]:
# Evaluate tuned model on validation set
best_model    = search.best_estimator_
y_proba_tuned = best_model.predict_proba(X_val)[:, 1]

# Evaluate at both default and optimal thresholds
for thr, label in [(0.5, 'Default (0.50)'), (optimal_threshold, f'Optimal ({optimal_threshold})')]:
    y_pred_t = (y_proba_tuned >= thr).astype(int)
    print(f"\nTuned LightGBM with {label} threshold:")
    print(f"  Accuracy  = {accuracy_score(y_val, y_pred_t):.4f}")
    print(f"  Precision = {precision_score(y_val, y_pred_t, zero_division=0):.4f}")
    print(f"  Recall    = {recall_score(y_val, y_pred_t, zero_division=0):.4f}")
    print(f"  F1        = {f1_score(y_val, y_pred_t, zero_division=0):.4f}")
    print(f"  ROC-AUC   = {roc_auc_score(y_val, y_proba_tuned):.4f}")

# Store optimal predictions for later
y_pred_tuned = (y_proba_tuned >= optimal_threshold).astype(int)

# Store in results dict for comparison
results['LightGBM (Tuned)'] = {
    'name'     : 'LightGBM (Tuned)',
    'pipeline' : best_model,
    'roc_auc'  : round(roc_auc_score(y_val, y_proba_tuned), 4),
    'f1'       : round(f1_score(y_val, y_pred_tuned, zero_division=0), 4),
    'recall'   : round(recall_score(y_val, y_pred_tuned, zero_division=0), 4),
    'precision': round(precision_score(y_val, y_pred_tuned, zero_division=0), 4),
    'accuracy' : round(accuracy_score(y_val, y_pred_tuned), 4),
    'y_pred'   : y_pred_tuned,
    'y_proba'  : y_proba_tuned,
}

In [ ]:
# Confusion matrices: default vs optimal threshold
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (thr, label) in zip(axes, [
    (0.5,              'Default threshold = 0.50'),
    (optimal_threshold, f'Optimal threshold = {optimal_threshold}')
]):
    y_pred_t = (y_proba_tuned >= thr).astype(int)
    cm = confusion_matrix(y_val, y_pred_t)
    ConfusionMatrixDisplay(cm, display_labels=['No Churn','Churn']).plot(
        ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'LightGBM Tuned — {label}', fontsize=11, fontweight='bold')

plt.suptitle('Effect of Threshold Optimisation on Confusion Matrix',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Confusion matrix key:")
print("  Top-left  (TN): Correctly predicted No Churn  ✅")
print("  Top-right (FP): Predicted Churn but actually stayed  → wasted retention offer")
print("  Bot-left  (FN): Predicted No Churn but actually churned  → missed! ❌ Worst outcome")
print("  Bot-right (TP): Correctly predicted Churn  ✅ Retention team can act")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Section 10.2 — Focused XGBoost Tuning with GridSearchCV
# ══════════════════════════════════════════════════════════════
from sklearn.model_selection import GridSearchCV, StratifiedKFold

imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()

xgb_tuning_pipeline = Pipeline([
    ('pre', preprocessor),
    ('clf', XGBClassifier(
        eval_metric='logloss',
        random_state=42,
        verbosity=0,
        n_jobs=-1
    ))
])

# ✏️ FIX 1: Test 2 values of scale_pos_weight to actually tune it
param_grid = {
    'clf__n_estimators':     [200, 300],
    'clf__max_depth':        [4, 6],
    'clf__learning_rate':    [0.03, 0.05],
    'clf__subsample':        [0.8, 1.0],
    'clf__colsample_bytree': [0.8, 1.0],
    'clf__scale_pos_weight': [imbalance_ratio, imbalance_ratio * 1.2],
}

grid_search = GridSearchCV(
    xgb_tuning_pipeline,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("Starting focused XGBoost GridSearchCV...")
grid_search.fit(X_train, y_train)

print("\nGrid search complete.")
print(f"Best CV AUC: {grid_search.best_score_:.4f}")
print("Best parameters:")
for k, v in grid_search.best_params_.items():
    print(f"  {k.replace('clf__', '')}: {v}")

best_xgb_model = grid_search.best_estimator_
y_proba_xgb_tuned = best_xgb_model.predict_proba(X_val)[:, 1]

# ✏️ FIX 2: Evaluate at BOTH thresholds (default AND optimal)
print("\n" + "="*55)
print("  XGBoost Tuned — Validation Results")
print("="*55)

for thr, label in [(0.5, 'Default (0.50)'), (optimal_threshold, f'Optimal ({optimal_threshold})')]:
    y_pred = (y_proba_xgb_tuned >= thr).astype(int)
    print(f"\n  Threshold = {label}:")
    print(f"    Accuracy  = {accuracy_score(y_val, y_pred):.4f}")
    print(f"    Precision = {precision_score(y_val, y_pred, zero_division=0):.4f}")
    print(f"    Recall    = {recall_score(y_val, y_pred, zero_division=0):.4f}")
    print(f"    F1        = {f1_score(y_val, y_pred, zero_division=0):.4f}")
    print(f"    ROC-AUC   = {roc_auc_score(y_val, y_proba_xgb_tuned):.4f}")

# ✏️ FIX 3: Compare against existing best model
xgb_auc = round(roc_auc_score(y_val, y_proba_xgb_tuned), 4)
current_best_auc = max(r['roc_auc'] for r in results.values())

print(f"\n{'='*55}")
print(f"  COMPARISON")
print(f"{'='*55}")
print(f"  Previous best AUC : {current_best_auc:.4f}")
print(f"  XGBoost Grid AUC  : {xgb_auc:.4f}")
print(f"  Difference        : {xgb_auc - current_best_auc:+.4f}")

# Store result in results dict
y_pred_optimal = (y_proba_xgb_tuned >= optimal_threshold).astype(int)
results['XGBoost (Grid Tuned)'] = {
    'name':      'XGBoost (Grid Tuned)',
    'pipeline':  best_xgb_model,
    'roc_auc':   xgb_auc,
    'accuracy':  round(accuracy_score(y_val, y_pred_optimal), 4),
    'precision': round(precision_score(y_val, y_pred_optimal, zero_division=0), 4),
    'recall':    round(recall_score(y_val, y_pred_optimal, zero_division=0), 4),
    'f1':        round(f1_score(y_val, y_pred_optimal, zero_division=0), 4),
    'y_pred':    y_pred_optimal,
    'y_proba':   y_proba_xgb_tuned,
}

# The final model is chosen explicitly in Section 10.4
print("\n  → Final model selection happens in Section 10.4")

In [ ]:
# ══════════════════════════════════════════════════════════════
# Section 10.3 — Ensemble: Soft Voting
# ══════════════════════════════════════════════════════════════
# WHY: Different models learn different patterns from the same data.
# Averaging their probability predictions produces a more stable,
# more accurate result than any single model alone.
#
# METHOD: Soft Voting — averages predicted PROBABILITIES, not hard labels.
# This is better than hard voting because it preserves confidence levels.
# A customer with 0.75 churn probability from all 3 models is very different
# from a customer with 0.51 from one model and 0.49 from two others.
# ══════════════════════════════════════════════════════════════

from sklearn.ensemble import VotingClassifier

# ── Step 1: Get the best version of each model type ──────────
# Use the classifiers that were already trained in Section 8
# Each one sits inside its own pipeline with preprocessing

ensemble_pipeline = Pipeline([
    ('pre', ColumnTransformer(transformers=[
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                          ('sc',  StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                          ('enc', OrdinalEncoder(handle_unknown='use_encoded_value',
                                                 unknown_value=-1))]), categorical_cols)
    ])),
    ('clf', VotingClassifier(
        estimators=[
            ('lgbm', LGBMClassifier(
                class_weight='balanced', n_estimators=300,
                max_depth=6, learning_rate=0.05,
                random_state=42, verbose=-1)),
            ('xgb', XGBClassifier(
                scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
                n_estimators=300, max_depth=6, learning_rate=0.05,
                subsample=0.8, colsample_bytree=0.8,
                random_state=42, eval_metric='logloss', verbosity=0)),
            ('rf', RandomForestClassifier(
                class_weight='balanced', n_estimators=300,
                max_depth=12, n_jobs=-1, random_state=42)),
        ],
        voting='soft',    # average probabilities (not hard labels)
        n_jobs=-1
    ))
])

print("Training Ensemble (LightGBM + XGBoost + Random Forest)...")
print("This may take 2-4 minutes...\n")
ensemble_pipeline.fit(X_train, y_train)
print("✅ Ensemble trained!")

# ── Step 2: Evaluate at both thresholds ──────────────────────
y_proba_ensemble = ensemble_pipeline.predict_proba(X_val)[:, 1]

y_pred_ens_default  = (y_proba_ensemble >= 0.5).astype(int)
y_pred_ens_optimal  = (y_proba_ensemble >= optimal_threshold).astype(int)

print(f"\nEnsemble (threshold=0.50):")
print(f"  Accuracy  = {accuracy_score(y_val, y_pred_ens_default):.4f}")
print(f"  Precision = {precision_score(y_val, y_pred_ens_default, zero_division=0):.4f}")
print(f"  Recall    = {recall_score(y_val, y_pred_ens_default, zero_division=0):.4f}")
print(f"  F1        = {f1_score(y_val, y_pred_ens_default, zero_division=0):.4f}")
print(f"  ROC-AUC   = {roc_auc_score(y_val, y_proba_ensemble):.4f}")

print(f"\nEnsemble (threshold={optimal_threshold}):")
print(f"  Accuracy  = {accuracy_score(y_val, y_pred_ens_optimal):.4f}")
print(f"  Precision = {precision_score(y_val, y_pred_ens_optimal, zero_division=0):.4f}")
print(f"  Recall    = {recall_score(y_val, y_pred_ens_optimal, zero_division=0):.4f}")
print(f"  F1        = {f1_score(y_val, y_pred_ens_optimal, zero_division=0):.4f}")
print(f"  ROC-AUC   = {roc_auc_score(y_val, y_proba_ensemble):.4f}")

# ── Step 3: Compare against best single model ────────────────
tuned_auc = results['LightGBM (Tuned)']['roc_auc']
ens_auc   = round(roc_auc_score(y_val, y_proba_ensemble), 4)

tuned_f1  = results['LightGBM (Tuned)']['f1']
ens_f1    = round(f1_score(y_val, y_pred_ens_optimal, zero_division=0), 4)

tuned_rec = results['LightGBM (Tuned)']['recall']
ens_rec   = round(recall_score(y_val, y_pred_ens_optimal, zero_division=0), 4)

tuned_acc = results['LightGBM (Tuned)']['accuracy']
ens_acc   = round(accuracy_score(y_val, y_pred_ens_optimal), 4)

print(f"\n{'='*60}")
print(f"  ENSEMBLE vs BEST SINGLE MODEL")
print(f"{'='*60}")
print(f"  {'Metric':<12} {'LightGBM Tuned':>16} {'Ensemble':>16} {'Diff':>10}")
print(f"  {'-'*54}")
print(f"  {'Accuracy':<12} {tuned_acc:>16.4f} {ens_acc:>16.4f} {ens_acc-tuned_acc:>+10.4f}")
print(f"  {'Recall':<12} {tuned_rec:>16.4f} {ens_rec:>16.4f} {ens_rec-tuned_rec:>+10.4f}")
print(f"  {'F1':<12} {tuned_f1:>16.4f} {ens_f1:>16.4f} {ens_f1-tuned_f1:>+10.4f}")
print(f"  {'ROC-AUC':<12} {tuned_auc:>16.4f} {ens_auc:>16.4f} {ens_auc-tuned_auc:>+10.4f}")

# ── Step 4: Store result (final selection happens in Section 10.4) ──
results['Ensemble'] = {
    'name': 'Ensemble (LightGBM+XGBoost+RF)', 'pipeline': ensemble_pipeline,
    'roc_auc': ens_auc, 'f1': ens_f1, 'recall': ens_rec, 'accuracy': ens_acc,
    'precision': round(precision_score(y_val, y_pred_ens_optimal, zero_division=0), 4),
    'y_pred': y_pred_ens_optimal, 'y_proba': y_proba_ensemble,
}

### 10.4 — Final Model Selection

The tuned LightGBM, the grid-tuned XGBoost and the soft-voting ensemble differ only in the third decimal place of
validation ROC-AUC — well within the noise of a single train/validation split.
Because **recall** is the business-critical metric and the ensemble loses recall at the chosen threshold,
and because LightGBM trains faster than XGBoost (easier periodic retraining),
the **tuned LightGBM** is selected as the final model.

In [ ]:
# Final model: tuned LightGBM from RandomizedSearchCV
best_model    = search.best_estimator_
y_proba_tuned = best_model.predict_proba(X_val)[:, 1]

summary = pd.DataFrame([
    {'Model': k, **{m: results[k][m] for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']}}
    for k in ['LightGBM (Tuned)', 'XGBoost (Grid Tuned)', 'Ensemble']
]).set_index('Model')
print(f"Validation metrics at threshold = {optimal_threshold}:")
display(summary)
print("\n✅ Final model: LightGBM (Tuned)")

---
## Section 11 — Model Interpretation: Feature Importance & SHAP

### 📖 What and Why

A model's accuracy is not enough — you need to understand **why** it makes predictions.

| Method | What it shows | When to use |
|--------|-------------|-------------|
| **Built-in importance** | Total loss reduction (gain) from splits on each feature | Quick first look |
| **Permutation importance** | AUC drop when feature is randomly shuffled | More reliable, threshold-free |
| **SHAP values** | Exactly how each feature pushed each prediction | Deepest insight, local + global |

In [ ]:
# 12.1 — Built-in LightGBM feature importance
clf_step     = best_model.named_steps['clf']
feature_names = numeric_cols + categorical_cols

importance_df = pd.DataFrame({
    'feature'   : feature_names,
    # normalised split gain (total loss reduction contributed by each feature)
    'importance': clf_step.booster_.feature_importance(importance_type='gain')
                  / clf_step.booster_.feature_importance(importance_type='gain').sum()
}).sort_values('importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 9))
top25   = importance_df.head(25)
colors  = ['#EF4444' if i < 5 else '#3B82F6' for i in range(len(top25))]
ax.barh(top25['feature'][::-1], top25['importance'][::-1],
        color=colors[::-1], edgecolor='white', alpha=0.85)
ax.set_xlabel('Feature Importance (Split Gain)', fontsize=12)
ax.set_title('Top 25 Feature Importances — LightGBM (Tuned)\n'
             '(🔴 Top 5 most important drivers)', fontsize=13, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("Top 10 features by built-in importance:")
for _, row in importance_df.head(10).iterrows():
    print(f"  {row['feature']:<35}  {row['importance']:>8.4f}")

In [ ]:
# 12.2 — Permutation importance
# For each feature: randomly shuffle its values → measure how much AUC drops
# Large drop = feature is important for generalisation
# Advantage over built-in: not biased toward high-cardinality features

print("Computing permutation importances (10 repeats × all features)...")
print("This takes ~2-4 minutes...\n")

X_val_t = best_model.named_steps['pre'].transform(X_val)

perm_result = permutation_importance(
    best_model.named_steps['clf'],
    X_val_t, y_val,
    n_repeats=10,
    random_state=42,
    scoring='roc_auc',
    n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature'   : feature_names,
    'importance': perm_result.importances_mean,
    'std'       : perm_result.importances_std
}).sort_values('importance', ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(10, 8))
top20 = perm_df.head(20)

# Top 5 features red, the rest blue — consistent with Figure 8
colors = ['#EF4444' if i < 5 else '#3B82F6' for i in range(len(top20))]

ax.barh(top20['feature'][::-1], top20['importance'][::-1],
        xerr=top20['std'][::-1], color=colors[::-1],
        error_kw={'elinewidth':1.2,'alpha':0.7}, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean AUC Decrease when Feature Shuffled', fontsize=12)
ax.set_title('Top 20 Features — Permutation Importance\n'
             '(Top 5 most important drivers in red | error bars = std. dev. over 10 repeats)',
             fontsize=12, fontweight='bold')
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\nTop 10 features (permutation importance):")
for _, row in perm_df.head(10).iterrows():
    bar = '█' * max(0, int(row['importance'] * 250))
    print(f"  {row['feature']:<35}  {row['importance']:+.4f} ± {row['std']:.4f}  {bar}")

In [ ]:
# 12.3 — SHAP: global summary (beeswarm plot)
# SHAP values explain EVERY prediction:
# For each customer: which features pushed toward "Churn" and by how much?
# The beeswarm plot shows the distribution of these effects across all customers.

print("Computing SHAP values (sample of 2000 validation rows)...")
sample_idx   = np.random.default_rng(42).choice(len(X_val_t), 2000, replace=False)
X_val_sample = X_val_t[sample_idx]

explainer   = shap.TreeExplainer(best_model.named_steps['clf'])
shap_values = explainer.shap_values(X_val_sample)

# For binary classification, shap_values may be [class0_values, class1_values]
shap_class1 = shap_values[1] if isinstance(shap_values, list) else shap_values

print(f"✅ SHAP computed for {len(X_val_sample)} customers")

plt.figure(figsize=(12, 9))
shap.summary_plot(shap_class1, X_val_sample,
                  feature_names=feature_names, show=False, max_display=20)
plt.title('SHAP Summary Plot — Top 20 Churn Drivers\n'
          '(Red = high feature value  |  Blue = low feature value)',
          fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

print("\nHow to read the SHAP beeswarm plot:")
print("  • Features at TOP = biggest average impact on predictions")
print("  • Dot to RIGHT (+) = pushes this customer toward Churn")
print("  • Dot to LEFT  (−) = pushes this customer away from Churn")
print("  • RED dot = high value of that feature | BLUE dot = low value")

In [ ]:
# 12.4 — SHAP waterfall: explain one individual customer
# Shows exactly how each feature moved ONE customer's prediction
# from the average (base value) up to their final prediction

churn_indices = np.where(y_val.values[sample_idx] == 1)[0]
example_idx   = churn_indices[0]

base_val = (explainer.expected_value[1]
            if isinstance(explainer.expected_value, list)
            else explainer.expected_value)

shap_exp = shap.Explanation(
    values       = shap_class1[example_idx],
    base_values  = base_val,
    data         = X_val_sample[example_idx],
    feature_names= feature_names
)

plt.figure(figsize=(12, 7))
shap.plots.waterfall(shap_exp, max_display=15, show=False)
plt.title('SHAP Waterfall — Single Customer (Actual Label: Churn)',
          fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nHow to read the waterfall plot:")
print("  E[f(x)] at bottom = average model prediction across ALL customers")
print("  f(x) at top       = final prediction for THIS specific customer")
print("  RED bars  = features increasing this customer's churn probability")
print("  BLUE bars = features decreasing this customer's churn probability")
print("  Numbers on right  = actual feature values for this customer")

In [ ]:
# 12.5 — Business translation of top SHAP drivers
print("=" * 65)
print("  BUSINESS INTERPRETATION OF TOP CHURN DRIVERS")
print("=" * 65)

insights = [
    ("CurrentEquipmentDays", "DEVICE AGE",
     "Customers with older phones churn more. They want an upgrade but haven't been offered one proactively."),
    ("MonthsInService", "TENURE",
     "New customers churn early (not yet loyal). Long-tenure customers are sticky. Mid-tenure is risky."),
    ("PercChangeMinutes", "USAGE DROP",
     "Sudden decline in call volume is an early warning. Disengagement precedes cancellation."),
    ("MonthlyRevenue", "MONTHLY BILL",
     "Higher charges increase churn. Price-sensitive customers compare plans and leave for cheaper options."),
    ("RetentionCalls", "ALREADY IN RETENTION PIPELINE",
     "Customers who called retention are highest-risk. Escalate immediately with the best available offer."),
    ("CustomerCareCalls", "FRUSTRATION SIGNAL",
     "Repeated care calls = unresolved issues. Every unanswered complaint increases churn probability."),
    ("TotalRecurringCharge", "FIXED MONTHLY COST",
     "High fixed charges drive price sensitivity. Consider loyalty discounts for at-risk high-value customers."),
    ("CreditRating", "FINANCIAL PROFILE",
     "Lower credit scores correlate with higher churn — may indicate less committed or prepaid customers."),
]

for feat, category, explanation in insights:
    if feat in perm_df.head(15)['feature'].values:
        print(f"\n  {category}  [{feat}]")
        import textwrap
        for line in textwrap.wrap(explanation, 60):
            print(f"    {line}")

---
## Section 12 — Final Holdout Predictions ⚠️

### 📖 What and Why

This is the moment of truth. Everything until now used only `df_train`.

We now apply the **final trained pipeline to holdout data exactly once**.

> **Rules for this section:**
> - No refitting — the pipeline is used exactly as trained
> - No threshold changes — we use `optimal_threshold` found on validation set
> - The holdout has no Churn labels — we generate predictions only
> - We check if the prediction distribution matches the validation distribution (overfitting check)

This simulates real-world deployment: train once, predict on new customers forever.

In [ ]:
# Apply the SAME cleaning function (is_train=False — no Churn encoding needed)
df_holdout_clean, _ = clean(df_holdout, is_train=False)

print(f"Holdout cleaned: {df_holdout_clean.shape}")

# Re-apply the SAME feature engineering function used for training
X_holdout_fe = add_features(df_holdout_clean)

# Align column order to match training exactly and select only the features used for training
X_holdout = X_holdout_fe[X.columns]

print(f"Columns match training feature set: {set(X_holdout.columns) == set(X.columns)}")
print(f"\n✅ Holdout ready: {X_holdout.shape[0]:,} rows × {X_holdout.shape[1]} features")
print(f"   Final model : LightGBM (Tuned via RandomizedSearchCV)")
print(f"   Threshold   : {optimal_threshold}")

In [ ]:
# Generate predictions — pipeline applies fitted preprocessor then model
# NO refitting happens here
y_proba_holdout = best_model.predict_proba(X_holdout)[:, 1]
y_pred_holdout  = (y_proba_holdout >= optimal_threshold).astype(int)

n_churn    = y_pred_holdout.sum()
pct_churn  = n_churn / len(y_pred_holdout) * 100

print("HOLDOUT PREDICTION RESULTS")
print("=" * 50)
print(f"  Total customers          : {len(y_pred_holdout):>7,}")
print(f"  Predicted to CHURN       : {n_churn:>7,}  ({pct_churn:.1f}%)")
print(f"  Predicted to STAY        : {len(y_pred_holdout)-n_churn:>7,}  ({100-pct_churn:.1f}%)")
print()
print("  These are the customers your retention team should contact!")
print()
print("Churn probability distribution:")
bins = pd.cut(y_proba_holdout, bins=[0,.2,.4,.6,.8,1.0],
              labels=['0-20%','20-40%','40-60%','60-80%','80-100%'])
print(bins.value_counts().sort_index().to_frame('customer_count'))

In [ ]:
# Distribution plot: validation vs holdout probability
# If distributions are similar → model generalised well (no overfitting)
# If very different → possible data shift or overfitting problem

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(y_proba_tuned,   bins=50, alpha=0.6, color='#3B82F6',
             density=True, label='Validation probabilities')
axes[0].hist(y_proba_holdout, bins=50, alpha=0.6, color='#EF4444',
             density=True, label='Holdout probabilities')
axes[0].axvline(optimal_threshold, color='black', linestyle='--', lw=1.5,
                label=f'Threshold = {optimal_threshold}')
axes[0].set_xlabel('Predicted Churn Probability')
axes[0].set_ylabel('Density')
axes[0].set_title('Validation vs Holdout Probability Distribution\n'
                  '(Close overlap → good generalisation)', fontsize=11, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].pie([len(y_pred_holdout)-n_churn, n_churn],
            labels=['Predicted No Churn','Predicted Churn'],
            autopct='%1.1f%%', colors=['#3B82F6','#EF4444'],
            startangle=90, wedgeprops=dict(edgecolor='white', lw=2))
axes[1].set_title('Holdout Prediction Breakdown', fontsize=11, fontweight='bold')

plt.suptitle('Final Holdout Predictions — Unseen Data',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

diff = abs(y_proba_tuned.mean() - y_proba_holdout.mean())
print(f"\nMean predicted probability — Validation: {y_proba_tuned.mean():.4f}"
      f"  |  Holdout: {y_proba_holdout.mean():.4f}  |  Δ = {diff:.4f}")
if diff < 0.03:
    print("✅ Very similar → model generalises well, no major overfitting detected")
elif diff < 0.06:
    print("⚠️  Slight difference → minor distribution shift, still acceptable")
else:
    print("🚨 Large difference → investigate for overfitting or data shift!")

In [ ]:
# Export predictions with risk tiers for the business team
predictions = df_holdout[['CustomerID']].copy()
predictions['ChurnProbability'] = y_proba_holdout.round(4)
predictions['ChurnPredicted']   = y_pred_holdout
predictions['RiskTier']         = pd.cut(
    y_proba_holdout,
    bins=[0, 0.3, 0.5, 0.7, 1.0],
    labels=['Low Risk','Medium Risk','High Risk','Critical Risk']
)
predictions = predictions.sort_values('ChurnProbability', ascending=False)

print("Top 15 highest-risk customers:\n")
display(predictions.head(15))

predictions.to_csv(OUTPUT_DIR / 'churn_predictions_holdout.csv', index=False)
print(f"\n✅ Saved to outputs/churn_predictions_holdout.csv")
print(f"   {n_churn:,} customers flagged for retention outreach")

---
## Section 13 — Final Summary, Business Recommendations & Key Lessons

In [ ]:
# Final model scorecard
lgbm_met = results['LightGBM (Tuned)']

print("╔" + "═"*64 + "╗")
print("║" + " FINAL MODEL SCORECARD ".center(64) + "║")
print("╠" + "═"*64 + "╣")
print(f"║  Algorithm      :  LightGBM (Tuned via RandomizedSearchCV)    ║")
print(f"║  Training rows  :  {len(X_train):,}                                       ║")
print(f"║  Decision threshold: {optimal_threshold}  (optimised for F1 on val set)       ║")
print("╠" + "═"*64 + "╣")
print("║  VALIDATION SET METRICS                                      ║")
print(f"║    Accuracy   = {lgbm_met['accuracy']:.4f}                                       ║")
print(f"║    Precision  = {lgbm_met['precision']:.4f}                                       ║")
print(f"║    Recall     = {lgbm_met['recall']:.4f}  ← most important metric for churn    ║")
print(f"║    F1-Score   = {lgbm_met['f1']:.4f}                                       ║")
print(f"║    ROC-AUC    = {lgbm_met['roc_auc']:.4f}  ← overall discrimination ability    ║")
print("╠" + "═"*64 + "╣")
print(f"║  HOLDOUT PREDICTIONS                                         ║")
print(f"║    Holdout customers    : {len(y_pred_holdout):,}                               ║")
print(f"║    Flagged as churners  : {n_churn:,}  ({pct_churn:.1f}%)                          ║")
print("╚" + "═"*64 + "╝")

In [ ]:
# Business recommendations
print("ACTIONABLE BUSINESS RECOMMENDATIONS")
print("=" * 65)

recs = [
    ("🎯 Priority 1 — Proactive Device Upgrade Program",
     ["CurrentEquipmentDays is the #1 churn driver",
      "Offer upgrades to customers with devices older than 400 days",
      "Frame as a loyalty reward, not a sales call"]),
    ("📞 Priority 2 — Retention Escalation Protocol",
     ["Customers who called retention are highest-risk",
      "Escalate to senior specialist with a custom, personalised offer"]),
    ("📉 Priority 3 — Usage Drop Alert System",
     ["Set automated alerts when usage falls >30% month-over-month",
      "Trigger proactive outreach before the customer decides to leave",
      "PercChangeMinutes is a powerful early warning signal"]),
    ("💰 Priority 4 — Price-Sensitive Loyalty Discounts",
     ["High-paying customers with declining usage = highest financial risk",
      "Offer loyalty pricing to top-revenue-quartile customers showing risk signals",
      "Consider family plan bundles to increase switching costs"]),
    ("🎧 Priority 5 — Customer Care Quality",
     ["CustomerCareCalls predicts churn — not because calling is bad",
      "Repeated calls signal UNRESOLVED issues",
      "Measure and improve first-call resolution rate"]),
    ("📊 Priority 6 — Risk-Tiered Outreach",
     ["Critical Risk (>70% prob) → personal call + best available offer",
      "High Risk (50–70%)       → targeted email/SMS with upgrade offer",
      "Medium Risk (30–50%)     → automated engagement campaign",
      "Low Risk (<30%)          → standard communications only"]),
]

import textwrap
for title, points in recs:
    print(f"\n  {title}")
    for p in points:
        print(f"    • {p}")

In [ ]:
# Limitations and future improvements
print("MODEL LIMITATIONS")
print("=" * 65)
limitations = [
    ("Moderate AUC (~0.67–0.68)",
     "Churn is hard to predict from behavioral data alone. Expect some false positives/negatives."),
    ("No temporal features",
     "We treat each customer as a static snapshot. Time-series trends could improve performance."),
    ("ServiceArea dropped",
     "Geographic location was removed due to high cardinality — regional dynamics are lost."),
    ("No competitive data",
     "Competitor pricing pressure is a major churn driver not captured in this dataset."),
    ("Static model",
     "Customer behaviour changes — model should be retrained quarterly on fresh data."),
]
for title, detail in limitations:
    print(f"\n  ⚠️  {title}")
    for line in textwrap.wrap(detail, 60):
        print(f"     {line}")

print("\n\nFUTURE IMPROVEMENTS")
print("=" * 65)
improvements = [
    "Add time-series features: rolling averages, month-over-month trends",
    "Use survival analysis (Cox model) to predict WHEN a customer churns",
    "Add external data: competitive pricing, network quality, NPS scores",
    "Encode ServiceArea via geographic clustering (group similar area codes)",
    "Implement model monitoring: track drift and performance decay monthly",
    "Test CatBoost which handles categorical features natively",
    "Use Optuna or Bayesian optimisation instead of RandomizedSearch",
    "Implement SHAP-based personalised retention messages per customer",
]
for i, item in enumerate(improvements, 1):
    print(f"  {i:>2}. {item}")

---
## 📚 Appendix — The Complete Workflow in Plain Language

1. **Loaded two datasets** — one for developing the model, one locked away for the final predictions
2. **Inspected the data** before touching it — shape, types, missing values, target
3. **Cleaned it** — removed irrelevant columns, fixed types, documented every action
4. **Explored it visually** — found which features relate to churn and which features are redundant
5. **Engineered ratio features** and removed highly correlated duplicates
6. **Built a preprocessing pipeline** that handles all transformations without data leakage
7. **Split the training data** into train (80%) and validation (20%) using stratified sampling
8. **Trained 4 model families** and compared them on 5 metrics on the same validation set
9. **Handled class imbalance** using class weights, threshold optimisation and SMOTE
10. **Tuned the models** with RandomizedSearchCV / GridSearchCV — cross-validation on training data only
11. **Interpreted the model** with gain importance, permutation importance and SHAP values
12. **Applied the final pipeline once** to the holdout data and generated risk-tiered predictions

---

## 🎓 Key Lessons from This Project

| Lesson | Why it matters |
|--------|---------------|
| **Accuracy is not enough** | A "predict all No" model gets 71% accuracy but catches zero churners |
| **Pipelines prevent leakage** | Imputers/scalers must fit on training data only — pipelines enforce this |
| **Gradient boosting fits tabular data** | XGBoost / LightGBM clearly beat the linear baseline |
| **The threshold is a business decision** | 0.5 is arbitrary — choose it from the cost of missed churners vs. false alarms |
| **Recall matters most for churn** | Missing a churner costs a customer; a false alarm costs only a small offer |
| **CV for tuning, validation for reporting** | Tuning on the validation set would bias the reported metrics |
| **Holdout is touched once** | That is what makes the final predictions credible |
| **Interpretability builds trust** | SHAP values explain predictions to non-technical stakeholders |

In [ ]:
# Final cell — project complete
print("=" * 67)
print("  ✅  CELL2CELL CHURN PREDICTION PROJECT — COMPLETE")
print("=" * 67)
print()
print("  Output file produced:")
print("    📄 outputs/churn_predictions_holdout.csv  — scored holdout with risk tiers")
print()
lgbm_m = results['LightGBM (Tuned)']
print(f"  Final model    :  LightGBM (Tuned via RandomizedSearchCV)")
print(f"  Val ROC-AUC    :  {lgbm_m['roc_auc']:.4f}")
print(f"  Val Recall     :  {lgbm_m['recall']:.4f}  (caught {lgbm_m['recall']*100:.1f}% of real churners)")
print(f"  Val F1-Score   :  {lgbm_m['f1']:.4f}")
print(f"  Threshold      :  {optimal_threshold}")
print(f"  Holdout flagged:  {n_churn:,} / {len(y_pred_holdout):,} customers for retention")
print()
print("  This notebook is fully reproducible.")
print("  Set random_state=42 everywhere and re-run from the top.")
print("=" * 67)